# 08 GHSL Chennai Metropolitan Production
Uses the frozen Hyderabad-validated baseline. Run one cell at a time. Cells 3-6 are gates; do not launch Cell 7 unless all pass.


In [1]:
# CELL 1: BOOTSTRAP
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess
drive.mount('/content/drive', force_remount=False)
D=Path('/content/drive/MyDrive'); R=D/'remote_exec.sh'; K=D/'hadoop-hdfs.pem'; L=Path('/content/hadoop-hdfs.pem')
if not R.is_file(): raise FileNotFoundError(R)
os.chmod(R,0o700)
if not L.is_file(): shutil.copy2(K,L)
os.chmod(L,0o600)
def run_remote(command,timeout=300,show_output=True):
    r=subprocess.run(['bash',str(R),command],text=True,capture_output=True,timeout=timeout)
    if show_output and r.stdout: print(r.stdout)
    if show_output and r.stderr: print('REMOTE STDERR:'); print(r.stderr)
    if r.returncode: raise RuntimeError(f'Remote command failed with return code {r.returncode}.')
    return r
print('CELL 1: SUCCESS')

Mounted at /content/drive
CELL 1: SUCCESS


In [2]:
# CELL 2: FROZEN CHENNAI CONFIGURATION

ACTIVE_CITY_SLUG = "chennai_metro"

ACTIVE_CITY = {
    "city_name": "Chennai Metropolitan Region",
    "city_slug": "chennai_metro",
    "seed_lon": 80.2707,
    "seed_lat": 13.0827,
    "target_crs": "ESRI:54009",
    "expected_grid_cells": 4900,
    "expected_complete_cells": 4900,
    "expected_partial_cells": 0,
    "extent_mollweide": {
        "min_x": 7880000.0,
        "min_y": 1577000.0,
        "max_x": 7950000.0,
        "max_y": 1647000.0,
    },
    "hdfs_output_base": (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/cities/"
        "city=chennai_metro"
    ),
    "remote_work_dir": (
        "/home/ubuntu/ghsl_multicity/chennai_metro"
    ),
}

assert (
    ACTIVE_CITY["extent_mollweide"]["max_x"]
    - ACTIVE_CITY["extent_mollweide"]["min_x"]
    == 70000
)

print(ACTIVE_CITY)
print("CELL 2: SUCCESS")

{'city_name': 'Chennai Metropolitan Region', 'city_slug': 'chennai_metro', 'seed_lon': 80.2707, 'seed_lat': 13.0827, 'target_crs': 'ESRI:54009', 'expected_grid_cells': 4900, 'expected_complete_cells': 4900, 'expected_partial_cells': 0, 'extent_mollweide': {'min_x': 7880000.0, 'min_y': 1577000.0, 'max_x': 7950000.0, 'max_y': 1647000.0}, 'hdfs_output_base': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=chennai_metro', 'remote_work_dir': '/home/ubuntu/ghsl_multicity/chennai_metro'}
CELL 2: SUCCESS


In [3]:
# ============================================================
# CELL 3: BASELINE AND HDFS PREFLIGHT
#
# Corrected behavior:
#   - Verifies the frozen SHA-256 inventory
#   - Compiles baseline Python files into /tmp
#   - Does not create __pycache__ in the baseline
#   - Removes bytecode created by the previous Cell 3 attempt
#   - Verifies all permanent baseline files are read-only
#
# Does not:
#   - modify baseline source files
#   - launch Spark
#   - modify HDFS
# ============================================================

cmd = r"""
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

BASE="/home/ubuntu/ghsl_multicity_baseline"
COMPILE_ROOT="/tmp/ghsl_baseline_compile_check"

echo "============================================================"
echo "CHENNAI BASELINE AND HDFS PREFLIGHT"
echo "============================================================"


# ------------------------------------------------------------
# HDFS service validation.
# ------------------------------------------------------------

for daemon in \
    NameNode \
    DataNode \
    SecondaryNameNode

do
    jps \
        | awk '{print $2}' \
        | grep -qx "$daemon"
done

hdfs dfsadmin -safemode get \
    | grep -qi "Safe mode is OFF"

echo "HDFS services: PASS"


# ------------------------------------------------------------
# Required baseline files.
# ------------------------------------------------------------

required_files=(
    "$BASE/baseline_manifest.json"
    "$BASE/SHA256SUMS"
    "$BASE/README.txt"
    "$BASE/source_city_config.json"
    "$BASE/batch/run_city_epochs.py"
    "$BASE/templates/population_epoch_template.py"
    "$BASE/templates/smod_epoch_template.py"
    "$BASE/templates/built_surface_epoch_template.py"
)

for file in "${required_files[@]}"; do
    if [ ! -f "$file" ]; then
        echo "ERROR: Required baseline file missing:"
        echo "$file"
        exit 20
    fi
done

echo "Required baseline files: PASS"


# ------------------------------------------------------------
# Verify the permanent baseline inventory before doing any
# compilation.
# ------------------------------------------------------------

cd "$BASE"

sha256sum -c SHA256SUMS

echo "Baseline SHA-256 inventory: PASS"


# ------------------------------------------------------------
# Remove only generated Python bytecode created by the earlier
# preflight attempt.
#
# No source, manifest, checksum, config, or README is removed.
# ------------------------------------------------------------

find "$BASE" \
    -type d \
    -name "__pycache__" \
    -print \
    -exec rm -rf {} + \
    2>/dev/null \
    || true

find "$BASE" \
    -type f \
    \( -name "*.pyc" -o -name "*.pyo" \) \
    -print \
    -delete \
    2>/dev/null \
    || true

echo "Generated baseline bytecode removed: PASS"


# ------------------------------------------------------------
# Compile outside the frozen baseline.
# ------------------------------------------------------------

rm -rf "$COMPILE_ROOT"

mkdir -p \
    "$COMPILE_ROOT/batch" \
    "$COMPILE_ROOT/templates"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile \
    "$BASE/batch/run_city_epochs.py"

for file in \
    "$BASE/templates/population_epoch_template.py" \
    "$BASE/templates/smod_epoch_template.py" \
    "$BASE/templates/built_surface_epoch_template.py"

do
    PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
    python3 -m py_compile "$file"
done

echo "Baseline Python compilation: PASS"


# ------------------------------------------------------------
# Confirm compilation created nothing inside the baseline.
# ------------------------------------------------------------

if find "$BASE" \
    -type d \
    -name "__pycache__" \
    | grep -q .
then
    echo "ERROR: Compilation created __pycache__ in baseline."
    exit 21
fi

if find "$BASE" \
    -type f \
    \( -name "*.pyc" -o -name "*.pyo" \) \
    | grep -q .
then
    echo "ERROR: Compilation created bytecode in baseline."
    exit 22
fi

echo "Baseline remains bytecode-free: PASS"


# ------------------------------------------------------------
# Verify all permanent baseline files are read-only.
#
# The baseline directories remain owner-writable so future
# city workspaces can be created elsewhere. Only files need
# to remain frozen.
# ------------------------------------------------------------

writable_files="$(
    find "$BASE" \
        -type f \
        -perm -u=w \
        -print
)"

if [ -n "$writable_files" ]; then
    echo "ERROR: Writable permanent baseline files found:"
    echo "$writable_files"
    exit 23
fi

echo "Baseline permanent files read-only: PASS"


# ------------------------------------------------------------
# Recheck the inventory after cleanup and compilation.
# ------------------------------------------------------------

cd "$BASE"

sha256sum -c SHA256SUMS

echo "Post-check SHA-256 inventory: PASS"


# ------------------------------------------------------------
# Clean temporary compilation output.
# ------------------------------------------------------------

rm -rf "$COMPILE_ROOT"

echo
echo "BASELINE_PREFLIGHT_SUCCESS"
"""

preflight_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 3 RESULT")
print("=" * 80)

print(
    "Return code:",
    preflight_result.returncode,
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("Baseline source files modified: NO")
print("CELL 3: SUCCESS")

CHENNAI BASELINE AND HDFS PREFLIGHT
HDFS services: PASS
Required baseline files: PASS
baseline_manifest.json: OK
batch/run_city_epochs.py: OK
source_city_config.json: OK
templates/built_surface_epoch_template.py: OK
templates/population_epoch_template.py: OK
templates/smod_epoch_template.py: OK
Baseline SHA-256 inventory: PASS
Generated baseline bytecode removed: PASS
Baseline Python compilation: PASS
Baseline remains bytecode-free: PASS
Baseline permanent files read-only: PASS
baseline_manifest.json: OK
batch/run_city_epochs.py: OK
source_city_config.json: OK
templates/built_surface_epoch_template.py: OK
templates/population_epoch_template.py: OK
templates/smod_epoch_template.py: OK
Post-check SHA-256 inventory: PASS

BASELINE_PREFLIGHT_SUCCESS

REMOTE STDERR:


CELL 3 RESULT
Return code: 0
Spark launched: NO
HDFS modified: NO
Baseline source files modified: NO
CELL 3: SUCCESS


In [4]:
# ============================================================
# CELL 4: CREATE CHENNAI WORKSPACE FROM FROZEN BASELINE
#
# Source:
#   /home/ubuntu/ghsl_multicity_baseline
#
# Destination:
#   /home/ubuntu/ghsl_multicity/chennai_metro
#
# Safety:
#   - Refuses to overwrite an existing partial workspace
#   - Removes its own partial workspace if this build fails
#   - Does not change the frozen baseline
#   - Does not change Hyderabad, Pune, or Bengaluru
#   - Does not launch Spark
#   - Does not modify HDFS
# ============================================================

import base64
import hashlib
import json
import textwrap


# ============================================================
# VALIDATE NOTEBOOK STATE
# ============================================================

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1 and 2."
        )


if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


expected_workspace = (
    "/home/ubuntu/ghsl_multicity/chennai_metro"
)

expected_hdfs_root = (
    "hdfs://localhost:9000"
    "/data/ghsl/curated/cities/"
    "city=chennai_metro"
)

expected_extent = {
    "min_x": 7880000.0,
    "min_y": 1577000.0,
    "max_x": 7950000.0,
    "max_y": 1647000.0,
}


if (
    ACTIVE_CITY["remote_work_dir"]
    != expected_workspace
):
    raise RuntimeError(
        "Unexpected Chennai workspace: "
        + ACTIVE_CITY["remote_work_dir"]
    )


if (
    ACTIVE_CITY["hdfs_output_base"]
    != expected_hdfs_root
):
    raise RuntimeError(
        "Unexpected Chennai HDFS root: "
        + ACTIVE_CITY["hdfs_output_base"]
    )


if (
    ACTIVE_CITY["city_name"]
    != "Chennai Metropolitan Region"
):
    raise RuntimeError(
        "Unexpected Chennai city name."
    )


if (
    ACTIVE_CITY["extent_mollweide"]
    != expected_extent
):
    raise RuntimeError(
        "Unexpected Chennai extent.\n"
        f"Expected: {expected_extent}\n"
        f"Actual:   "
        f"{ACTIVE_CITY['extent_mollweide']}"
    )


if (
    ACTIVE_CITY["expected_grid_cells"]
    != 4900
):
    raise RuntimeError(
        "Chennai expected_grid_cells must be 4900."
    )


# ============================================================
# SERIALIZE CHENNAI CONFIGURATION
# ============================================================

config_json = (
    json.dumps(
        ACTIVE_CITY,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

config_b64 = base64.b64encode(
    config_json.encode("utf-8")
).decode("ascii")


# ============================================================
# BUILD REMOTE WORKSPACE CREATOR
# ============================================================

remote_builder = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import sys
    import time


    BASELINE = Path(
        "/home/ubuntu/ghsl_multicity_baseline"
    )

    TARGET = Path(
        "/home/ubuntu/ghsl_multicity/chennai_metro"
    )

    CONFIG_PATH = Path(
        sys.argv[1]
    )

    TEMPLATE_NAMES = [
        "population_epoch_template.py",
        "smod_epoch_template.py",
        "built_surface_epoch_template.py",
    ]


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def compile_source(source, path):
        compile(
            source,
            str(path),
            "exec",
        )


    # ========================================================
    # LOAD CONFIGURATIONS
    # ========================================================

    if not CONFIG_PATH.is_file():
        raise FileNotFoundError(
            CONFIG_PATH
        )

    config = json.loads(
        CONFIG_PATH.read_text(
            encoding="utf-8"
        )
    )

    source_config_path = (
        BASELINE
        / "source_city_config.json"
    )

    if not source_config_path.is_file():
        raise FileNotFoundError(
            source_config_path
        )

    source_config = json.loads(
        source_config_path.read_text(
            encoding="utf-8"
        )
    )


    # ========================================================
    # VALIDATE CHENNAI CONFIGURATION
    # ========================================================

    expected_extent = {
        "min_x": 7880000.0,
        "min_y": 1577000.0,
        "max_x": 7950000.0,
        "max_y": 1647000.0,
    }

    if (
        config.get("city_slug")
        != "chennai_metro"
    ):
        raise RuntimeError(
            "Chennai configuration has the wrong slug."
        )

    if (
        config.get("city_name")
        != "Chennai Metropolitan Region"
    ):
        raise RuntimeError(
            "Chennai configuration has the wrong name."
        )

    if (
        config.get("remote_work_dir")
        != str(TARGET)
    ):
        raise RuntimeError(
            "Chennai configuration has the wrong "
            "remote workspace."
        )

    if (
        config.get("hdfs_output_base")
        != (
            "hdfs://localhost:9000"
            "/data/ghsl/curated/cities/"
            "city=chennai_metro"
        )
    ):
        raise RuntimeError(
            "Chennai configuration has the wrong "
            "HDFS output root."
        )

    if (
        config.get("expected_grid_cells")
        != 4900
    ):
        raise RuntimeError(
            "Chennai expected_grid_cells must be 4900."
        )

    if (
        config.get("extent_mollweide")
        != expected_extent
    ):
        raise RuntimeError(
            "Chennai configuration has the wrong extent."
        )


    # ========================================================
    # VALIDATE BASELINE INPUTS
    # ========================================================

    baseline_driver = (
        BASELINE
        / "batch"
        / "run_city_epochs.py"
    )

    baseline_manifest = (
        BASELINE
        / "baseline_manifest.json"
    )

    baseline_checksums = (
        BASELINE
        / "SHA256SUMS"
    )

    required_baseline_files = [
        baseline_driver,
        baseline_manifest,
        baseline_checksums,
        source_config_path,
    ]

    for required_path in (
        required_baseline_files
    ):
        if not required_path.is_file():
            raise FileNotFoundError(
                required_path
            )

    baseline_templates = {}

    for template_name in TEMPLATE_NAMES:
        template_path = (
            BASELINE
            / "templates"
            / template_name
        )

        if not template_path.is_file():
            raise FileNotFoundError(
                template_path
            )

        baseline_templates[
            template_name
        ] = template_path


    # ========================================================
    # CHECK FOR AN EXISTING WORKSPACE
    # ========================================================

    workspace_manifest = (
        TARGET
        / "chennai_workspace_manifest.json"
    )

    if TARGET.exists():
        if workspace_manifest.is_file():
            print("=" * 80)
            print("CHENNAI WORKSPACE CREATION")
            print("=" * 80)

            print(
                "Existing completed Chennai "
                "workspace found."
            )

            print(
                "Workspace:",
                TARGET,
            )

            print(
                "Existing workspace was not "
                "overwritten."
            )

            print(
                "CHENNAI_WORKSPACE_ALREADY_EXISTS"
            )

            print(
                "CHENNAI_WORKSPACE_BUILD_SUCCESS"
            )

            raise SystemExit(0)

        raise RuntimeError(
            "An incomplete Chennai workspace already "
            "exists and was not overwritten: "
            + str(TARGET)
        )


    # ========================================================
    # BUILD PARAMETER REPLACEMENTS
    # ========================================================

    source_root = str(
        source_config[
            "remote_work_dir"
        ]
    )

    target_root = str(
        config[
            "remote_work_dir"
        ]
    )

    source_slug = str(
        source_config[
            "city_slug"
        ]
    )

    target_slug = str(
        config[
            "city_slug"
        ]
    )

    source_name = str(
        source_config[
            "city_name"
        ]
    )

    target_name = str(
        config[
            "city_name"
        ]
    )

    source_extent = (
        source_config[
            "extent_mollweide"
        ]
    )

    target_extent = (
        config[
            "extent_mollweide"
        ]
    )

    replacements = [
        (
            source_root,
            target_root,
        ),
        (
            "city=" + source_slug,
            "city=" + target_slug,
        ),
        (
            source_slug,
            target_slug,
        ),
        (
            source_name,
            target_name,
        ),
    ]

    for key in [
        "min_x",
        "min_y",
        "max_x",
        "max_y",
    ]:
        replacements.append(
            (
                str(
                    float(
                        source_extent[key]
                    )
                ),
                str(
                    float(
                        target_extent[key]
                    )
                ),
            )
        )

        replacements.append(
            (
                str(
                    int(
                        source_extent[key]
                    )
                ),
                str(
                    int(
                        target_extent[key]
                    )
                ),
            )
        )


    def parameterize(source):
        output = source

        for old_value, new_value in replacements:
            output = output.replace(
                old_value,
                new_value,
            )

        return output


    # ========================================================
    # CREATE WORKSPACE
    # ========================================================

    try:
        (
            TARGET
            / "templates"
        ).mkdir(
            parents=True,
            exist_ok=False,
        )

        (
            TARGET
            / "programs"
        ).mkdir()

        (
            TARGET
            / "batch"
        ).mkdir()


        # ====================================================
        # PARAMETERIZE TEMPLATES
        # ====================================================

        template_hashes = {}

        for template_name in TEMPLATE_NAMES:
            source_path = (
                baseline_templates[
                    template_name
                ]
            )

            destination_path = (
                TARGET
                / "templates"
                / template_name
            )

            template_source = (
                source_path.read_text(
                    encoding="utf-8"
                )
            )

            template_source = parameterize(
                template_source
            )

            compile_source(
                template_source,
                destination_path,
            )

            destination_path.write_text(
                template_source,
                encoding="utf-8",
            )

            py_compile.compile(
                str(destination_path),
                doraise=True,
            )

            template_hashes[
                template_name
            ] = sha256_file(
                destination_path
            )


        # ====================================================
        # PARAMETERIZE DRIVER
        # ====================================================

        driver_destination = (
            TARGET
            / "batch"
            / "run_city_epochs.py"
        )

        driver_source = (
            baseline_driver.read_text(
                encoding="utf-8"
            )
        )

        driver_source = parameterize(
            driver_source
        )


        # ====================================================
        # REFRESH TEMPLATE CHECKSUMS THROUGH AST
        # ====================================================

        driver_tree = ast.parse(
            driver_source
        )

        checksum_node = None

        for node in driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target
                in node.targets
            ):
                checksum_node = node
                break

        if checksum_node is None:
            raise RuntimeError(
                "EXPECTED_TEMPLATE_SHA256 "
                "was not found."
            )

        new_checksum_dictionary = {
            "population": (
                template_hashes[
                    "population_epoch_template.py"
                ]
            ),
            "smod": (
                template_hashes[
                    "smod_epoch_template.py"
                ]
            ),
            "built_surface": (
                template_hashes[
                    "built_surface_epoch_template.py"
                ]
            ),
        }

        driver_lines = (
            driver_source.splitlines(
                keepends=True
            )
        )

        checksum_assignment = (
            "EXPECTED_TEMPLATE_SHA256 = "
            + repr(
                new_checksum_dictionary
            )
            + chr(10)
        )

        driver_lines[
            checksum_node.lineno - 1:
            checksum_node.end_lineno
        ] = [
            checksum_assignment
        ]

        driver_source = "".join(
            driver_lines
        )

        compile_source(
            driver_source,
            driver_destination,
        )

        driver_destination.write_text(
            driver_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(driver_destination),
            doraise=True,
        )


        # ====================================================
        # SAVE CONFIGURATION
        # ====================================================

        chennai_config_path = (
            TARGET
            / "city_config.json"
        )

        chennai_config_path.write_text(
            json.dumps(
                config,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


        # ====================================================
        # CODE-LEVEL PARAMETER AUDIT
        # ====================================================

        combined_source = (
            driver_source
        )

        for template_name in TEMPLATE_NAMES:
            combined_source += (
                TARGET
                / "templates"
                / template_name
            ).read_text(
                encoding="utf-8"
            )

        forbidden_values = [
            source_root,
            "city=" + source_slug,
            source_name,
        ]

        remaining_forbidden = [
            value
            for value in forbidden_values
            if value in combined_source
        ]

        if remaining_forbidden:
            raise RuntimeError(
                "Hyderabad leakage remains: "
                + repr(
                    remaining_forbidden
                )
            )

        required_code_values = [
            target_root,
            "city=" + target_slug,
            "7880000",
            "1577000",
            "7950000",
            "1647000",
        ]

        missing_code_values = [
            value
            for value
            in required_code_values
            if value
            not in combined_source
        ]

        if missing_code_values:
            raise RuntimeError(
                "Required Chennai code parameters "
                "are missing: "
                + repr(
                    missing_code_values
                )
            )


        # ====================================================
        # CONFIGURATION-LEVEL AUDIT
        # ====================================================

        saved_config = json.loads(
            chennai_config_path.read_text(
                encoding="utf-8"
            )
        )

        if (
            saved_config["city_name"]
            != "Chennai Metropolitan Region"
        ):
            raise RuntimeError(
                "Chennai city name is incorrect "
                "in city_config.json."
            )

        if (
            saved_config["city_slug"]
            != "chennai_metro"
        ):
            raise RuntimeError(
                "Chennai city slug is incorrect "
                "in city_config.json."
            )

        if (
            saved_config["extent_mollweide"]
            != expected_extent
        ):
            raise RuntimeError(
                "Chennai extent is incorrect "
                "in city_config.json."
            )


        # ====================================================
        # VERIFY CHECKSUM DICTIONARY FROM SAVED DRIVER
        # ====================================================

        saved_driver_tree = ast.parse(
            driver_destination.read_text(
                encoding="utf-8"
            )
        )

        saved_checksum_dictionary = None

        for node in saved_driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target
                in node.targets
            ):
                saved_checksum_dictionary = (
                    ast.literal_eval(
                        node.value
                    )
                )

                break

        if (
            saved_checksum_dictionary
            != new_checksum_dictionary
        ):
            raise RuntimeError(
                "Saved driver checksum dictionary "
                "does not match Chennai templates."
            )


        # ====================================================
        # REMOVE GENERATED BYTECODE
        # ====================================================

        for pycache in TARGET.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )


        # ====================================================
        # SAVE WORKSPACE MANIFEST
        # ====================================================

        manifest = {
            "city_slug": target_slug,
            "city_name": target_name,
            "source_baseline": str(
                BASELINE
            ),
            "workspace": str(
                TARGET
            ),
            "driver_sha256": sha256_file(
                driver_destination
            ),
            "template_sha256": (
                new_checksum_dictionary
            ),
            "extent_mollweide": (
                expected_extent
            ),
            "expected_grid_cells": 4900,
            "created_at_utc": (
                time.strftime(
                    "%Y-%m-%dT%H:%M:%SZ",
                    time.gmtime(),
                )
            ),
        }

        workspace_manifest.write_text(
            json.dumps(
                manifest,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


    except Exception:
        # Do not leave behind a partial Chennai workspace.
        if TARGET.exists():
            shutil.rmtree(
                TARGET
            )

        raise


    # ========================================================
    # FINAL REPORT
    # ========================================================

    print("=" * 80)
    print("CHENNAI WORKSPACE CREATION")
    print("=" * 80)

    print(
        "Workspace:",
        TARGET,
    )

    print(
        "Driver:",
        TARGET
        / "batch"
        / "run_city_epochs.py",
    )

    print(
        "Configuration:",
        TARGET
        / "city_config.json",
    )

    print(
        "Manifest:",
        workspace_manifest,
    )

    print()
    print("Template parameterization: PASS")
    print("Template compilation: PASS")
    print("Driver parameterization: PASS")
    print("Driver compilation: PASS")
    print("Template checksum refresh: PASS")
    print("Saved checksum verification: PASS")
    print("Hyderabad leakage audit: PASS")
    print("Chennai code-parameter audit: PASS")
    print("Chennai configuration audit: PASS")
    print("Chennai extent validation: PASS")
    print("Partial-build cleanup protection: PASS")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print("CHENNAI_WORKSPACE_BUILD_SUCCESS")
    '''
).strip() + "\n"


# ============================================================
# LOCAL BUILDER VALIDATION
# ============================================================

compile(
    remote_builder,
    "build_chennai_workspace.py",
    "exec",
)

builder_sha256 = hashlib.sha256(
    remote_builder.encode("utf-8")
).hexdigest()

print(
    "LOCAL CHENNAI BUILDER COMPILE: PASS"
)

print(
    "Expected builder SHA-256:",
    builder_sha256,
)


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

builder_b64 = base64.b64encode(
    remote_builder.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

BUILDER="/tmp/build_chennai_workspace.py"
CONFIG="/tmp/chennai_city_config.json"

EXPECTED_BUILDER_SHA256="{builder_sha256}"

echo "{builder_b64}" \
| base64 -d \
> "${{BUILDER}}"

echo "{config_b64}" \
| base64 -d \
> "${{CONFIG}}"

ACTUAL_BUILDER_SHA256="$(
    sha256sum "${{BUILDER}}" \
    | awk '{{print $1}}'
)"

echo "Remote builder SHA-256: ${{ACTUAL_BUILDER_SHA256}}"

if [ "${{ACTUAL_BUILDER_SHA256}}" != "${{EXPECTED_BUILDER_SHA256}}" ]; then
    echo "ERROR: Builder checksum mismatch."
    exit 20
fi

echo "REMOTE BUILDER CHECKSUM: PASS"

python3 -m py_compile "${{BUILDER}}"

echo "REMOTE BUILDER COMPILE: PASS"

python3 "${{BUILDER}}" "${{CONFIG}}"
"""


build_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 4 RESULT")
print("=" * 80)

print(
    "Return code:",
    build_result.returncode,
)

print(
    "Workspace:",
    ACTIVE_CITY["remote_work_dir"],
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 4: SUCCESS")

LOCAL CHENNAI BUILDER COMPILE: PASS
Expected builder SHA-256: 4ef49185e1ab2cbda72016a21b7ddec1ae57f1d984aa41bfcee30e38f4541f0b
Remote builder SHA-256: 4ef49185e1ab2cbda72016a21b7ddec1ae57f1d984aa41bfcee30e38f4541f0b
REMOTE BUILDER CHECKSUM: PASS
REMOTE BUILDER COMPILE: PASS
CHENNAI WORKSPACE CREATION
Workspace: /home/ubuntu/ghsl_multicity/chennai_metro
Driver: /home/ubuntu/ghsl_multicity/chennai_metro/batch/run_city_epochs.py
Configuration: /home/ubuntu/ghsl_multicity/chennai_metro/city_config.json
Manifest: /home/ubuntu/ghsl_multicity/chennai_metro/chennai_workspace_manifest.json

Template parameterization: PASS
Template compilation: PASS
Driver parameterization: PASS
Driver compilation: PASS
Template checksum refresh: PASS
Saved checksum verification: PASS
Hyderabad leakage audit: PASS
Chennai code-parameter audit: PASS
Chennai configuration audit: PASS
Chennai extent validation: PASS
Partial-build cleanup protection: PASS
Spark launched: NO
HDFS modified: NO
CHENNAI_WORKSPACE_BUILD_

In [5]:
# ============================================================
# CELL 5: CHENNAI STATIC AND LEAKAGE AUDIT
#
# This version deliberately avoids:
#   - outer f-strings around embedded Python
#   - fragile grep checks for Python spacing
#   - creating __pycache__ inside the workspace
#
# It verifies:
#   - required Chennai files
#   - Python compilation
#   - Chennai configuration
#   - driver functions
#   - 4,900-cell expectation
#   - template checksums
#   - workspace-manifest checksums
#   - Chennai paths and extent
#   - absence of Hyderabad leakage
#   - generalized endpoint logic
#   - generalized 2020 guards
#   - absence of stale launch state
#
# Read-only toward source files and HDFS.
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


remote_audit = r'''
set -e

ROOT="__CHENNAI_ROOT__"
DRIVER="$ROOT/batch/run_city_epochs.py"
CONFIG="$ROOT/city_config.json"
MANIFEST="$ROOT/chennai_workspace_manifest.json"

POPULATION_TEMPLATE="$ROOT/templates/population_epoch_template.py"
SMOD_TEMPLATE="$ROOT/templates/smod_epoch_template.py"
BUILT_TEMPLATE="$ROOT/templates/built_surface_epoch_template.py"

COMPILE_ROOT="/tmp/chennai_static_compile_check"


echo "============================================================"
echo "CHENNAI STATIC AND LEAKAGE AUDIT"
echo "============================================================"


# ============================================================
# REQUIRED FILES
# ============================================================

for required_file in \
    "$DRIVER" \
    "$CONFIG" \
    "$MANIFEST" \
    "$POPULATION_TEMPLATE" \
    "$SMOD_TEMPLATE" \
    "$BUILT_TEMPLATE"

do
    if [ ! -f "$required_file" ]; then
        echo "ERROR: Required file is missing:"
        echo "$required_file"
        exit 20
    fi
done

echo "Required Chennai files: PASS"


# ============================================================
# COMPILE OUTSIDE THE WORKSPACE
# ============================================================

rm -rf "$COMPILE_ROOT"

mkdir -p "$COMPILE_ROOT"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$DRIVER"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$POPULATION_TEMPLATE"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$SMOD_TEMPLATE"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$BUILT_TEMPLATE"

echo "Chennai Python compilation: PASS"


# ============================================================
# STRUCTURAL PYTHON AUDIT
#
# All paths are command-line arguments. No shell or notebook
# interpolation occurs inside this Python source.
# ============================================================

python3 - \
    "$ROOT" \
    "$DRIVER" \
    "$CONFIG" \
    "$MANIFEST" \
    "$POPULATION_TEMPLATE" \
    "$SMOD_TEMPLATE" \
    "$BUILT_TEMPLATE" <<'PY'
from pathlib import Path
import ast
import hashlib
import json
import sys


root = Path(sys.argv[1])
driver_path = Path(sys.argv[2])
config_path = Path(sys.argv[3])
manifest_path = Path(sys.argv[4])

population_template = Path(sys.argv[5])
smod_template = Path(sys.argv[6])
built_template = Path(sys.argv[7])


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


# ============================================================
# CONFIGURATION AUDIT
# ============================================================

config = json.loads(
    config_path.read_text(
        encoding="utf-8"
    )
)

expected_config = {
    "city_slug": "chennai_metro",
    "city_name": "Chennai Metropolitan Region",
    "remote_work_dir":
        "/home/ubuntu/ghsl_multicity/chennai_metro",
    "hdfs_output_base":
        "hdfs://localhost:9000"
        "/data/ghsl/curated/cities/"
        "city=chennai_metro",
    "expected_grid_cells": 4900,
    "expected_complete_cells": 4900,
    "expected_partial_cells": 0,
    "target_crs": "ESRI:54009",
}

for key, expected_value in expected_config.items():
    actual_value = config.get(key)

    if actual_value != expected_value:
        raise RuntimeError(
            "Configuration mismatch for "
            + key
            + ": expected "
            + repr(expected_value)
            + ", found "
            + repr(actual_value)
        )


expected_extent = {
    "min_x": 7880000.0,
    "min_y": 1577000.0,
    "max_x": 7950000.0,
    "max_y": 1647000.0,
}

actual_extent = config.get(
    "extent_mollweide"
)

if actual_extent != expected_extent:
    raise RuntimeError(
        "Chennai extent mismatch. Expected "
        + repr(expected_extent)
        + ", found "
        + repr(actual_extent)
    )

if (
    expected_extent["max_x"]
    - expected_extent["min_x"]
    != 70000.0
):
    raise RuntimeError(
        "Chennai X extent is not 70 km."
    )

if (
    expected_extent["max_y"]
    - expected_extent["min_y"]
    != 70000.0
):
    raise RuntimeError(
        "Chennai Y extent is not 70 km."
    )

print("Chennai configuration: PASS")


# ============================================================
# DRIVER STRUCTURE
# ============================================================

driver_source = driver_path.read_text(
    encoding="utf-8"
)

driver_tree = ast.parse(
    driver_source
)

function_names = set()

for node in driver_tree.body:
    if isinstance(
        node,
        ast.FunctionDef,
    ):
        function_names.add(
            node.name
        )


required_functions = {
    "generate_population_program",
    "generate_smod_program",
    "generate_built_program",
    "ensure_stage",
    "verify_template",
    "replace_exact_count",
}

missing_functions = sorted(
    required_functions
    - function_names
)

if missing_functions:
    raise RuntimeError(
        "Required driver functions are missing: "
        + repr(missing_functions)
    )

print("Required production functions: PASS")


# ============================================================
# TOP-LEVEL ASSIGNMENTS
# ============================================================

assignments = {}

for node in driver_tree.body:
    if not isinstance(
        node,
        ast.Assign,
    ):
        continue

    try:
        assignment_value = ast.literal_eval(
            node.value
        )
    except Exception:
        continue

    for target in node.targets:
        if isinstance(
            target,
            ast.Name,
        ):
            assignments[
                target.id
            ] = assignment_value


# Find a structural expected row/cell constant equal to 4900.

grid_expectations = {}

for name, value in assignments.items():
    uppercase_name = name.upper()

    if "EXPECTED" not in uppercase_name:
        continue

    if (
        "ROW" not in uppercase_name
        and "CELL" not in uppercase_name
    ):
        continue

    grid_expectations[name] = value


if 4900 not in grid_expectations.values():
    raise RuntimeError(
        "No expected row/cell constant equal "
        "to 4900 was found. Candidates: "
        + repr(grid_expectations)
    )

print(
    "Expected 4,900-cell constant: PASS",
    grid_expectations,
)


# ============================================================
# TEMPLATE CHECKSUM AUDIT
# ============================================================

driver_checksums = assignments.get(
    "EXPECTED_TEMPLATE_SHA256"
)

if not isinstance(
    driver_checksums,
    dict,
):
    raise RuntimeError(
        "EXPECTED_TEMPLATE_SHA256 is missing "
        "or is not a dictionary."
    )


actual_checksums = {
    "population": sha256_file(
        population_template
    ),
    "smod": sha256_file(
        smod_template
    ),
    "built_surface": sha256_file(
        built_template
    ),
}


if driver_checksums != actual_checksums:
    raise RuntimeError(
        "Driver template checksums do not match. "
        "Driver="
        + repr(driver_checksums)
        + ", actual="
        + repr(actual_checksums)
    )

print("Driver template checksums: PASS")


# ============================================================
# WORKSPACE MANIFEST AUDIT
# ============================================================

manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)

if (
    manifest.get("city_slug")
    != "chennai_metro"
):
    raise RuntimeError(
        "Workspace manifest has the wrong slug."
    )

if (
    manifest.get("city_name")
    != "Chennai Metropolitan Region"
):
    raise RuntimeError(
        "Workspace manifest has the wrong name."
    )

if (
    manifest.get("driver_sha256")
    != sha256_file(driver_path)
):
    raise RuntimeError(
        "Workspace driver checksum mismatch."
    )

if (
    manifest.get("template_sha256")
    != actual_checksums
):
    raise RuntimeError(
        "Workspace template checksum mismatch."
    )

print("Workspace manifest checksums: PASS")


# ============================================================
# CITY PATH AND LEAKAGE AUDIT
# ============================================================

combined_source = driver_source

combined_source += (
    population_template.read_text(
        encoding="utf-8"
    )
)

combined_source += (
    smod_template.read_text(
        encoding="utf-8"
    )
)

combined_source += (
    built_template.read_text(
        encoding="utf-8"
    )
)


required_chennai_values = [
    "/home/ubuntu/ghsl_multicity/chennai_metro",
    "city=chennai_metro",
    "7880000",
    "1577000",
    "7950000",
    "1647000",
]

missing_chennai_values = []

for value in required_chennai_values:
    if value not in combined_source:
        missing_chennai_values.append(
            value
        )

if missing_chennai_values:
    raise RuntimeError(
        "Required Chennai values are missing: "
        + repr(missing_chennai_values)
    )


forbidden_hyderabad_values = [
    "/home/ubuntu/ghsl_multicity/hyderabad",
    "city=hyderabad",
    "Hyderabad Metropolitan Region",
]

remaining_hyderabad_values = []

for value in forbidden_hyderabad_values:
    if value in combined_source:
        remaining_hyderabad_values.append(
            value
        )

if remaining_hyderabad_values:
    raise RuntimeError(
        "Hyderabad leakage remains: "
        + repr(
            remaining_hyderabad_values
        )
    )

print("Chennai paths and extent in code: PASS")
print("Hyderabad leakage audit: PASS")


# ============================================================
# GENERALIZED TEMPLATE LOGIC
# ============================================================

population_source = (
    population_template.read_text(
        encoding="utf-8"
    )
)

smod_source = (
    smod_template.read_text(
        encoding="utf-8"
    )
)

built_source = (
    built_template.read_text(
        encoding="utf-8"
    )
)


if (
    ".parquet(population_1975_path)"
    in population_source
):
    raise RuntimeError(
        "Population template still contains "
        "the obsolete 1975 self-reference."
    )

if (
    ".parquet(SMOD_1975_PATH)"
    in smod_source
):
    raise RuntimeError(
        "SMOD template still contains "
        "the obsolete 1975 self-reference."
    )

if (
    "product=population/"
    not in built_source
):
    raise RuntimeError(
        "Built template does not contain "
        "the population-grid reference."
    )

print("Generalized template logic: PASS")


# ============================================================
# GENERALIZED 2020 GUARDS
# ============================================================

def is_epoch_not_2020(node):
    return (
        isinstance(
            node,
            ast.Compare,
        )
        and isinstance(
            node.left,
            ast.Name,
        )
        and node.left.id == "epoch"
        and len(node.ops) == 1
        and isinstance(
            node.ops[0],
            ast.NotEq,
        )
        and len(
            node.comparators
        ) == 1
        and isinstance(
            node.comparators[0],
            ast.Constant,
        )
        and node.comparators[0].value
        == 2020
    )


def is_active_2020_check(node):
    return (
        isinstance(
            node,
            ast.Compare,
        )
        and isinstance(
            node.left,
            ast.Constant,
        )
        and node.left.value
        == "EPOCH = 2020"
        and len(node.ops) == 1
        and isinstance(
            node.ops[0],
            ast.In,
        )
        and len(
            node.comparators
        ) == 1
        and isinstance(
            node.comparators[0],
            ast.Name,
        )
        and node.comparators[0].id
        == "program"
    )


guards_checked = 0

for node in ast.walk(
    driver_tree
):
    if not isinstance(
        node,
        ast.If,
    ):
        continue

    messages = []

    for child in ast.walk(node):
        if (
            isinstance(
                child,
                ast.Constant,
            )
            and isinstance(
                child.value,
                str,
            )
        ):
            messages.append(
                child.value
            )

    if not any(
        "retains active 2020"
        in message
        for message in messages
    ):
        continue

    if not (
        isinstance(
            node.test,
            ast.BoolOp,
        )
        and isinstance(
            node.test.op,
            ast.And,
        )
    ):
        raise RuntimeError(
            "A retained-active-2020 guard "
            "is not generalized."
        )

    has_epoch_condition = any(
        is_epoch_not_2020(
            condition
        )
        for condition
        in node.test.values
    )

    has_program_condition = any(
        is_active_2020_check(
            condition
        )
        for condition
        in node.test.values
    )

    if not (
        has_epoch_condition
        and has_program_condition
    ):
        raise RuntimeError(
            "A 2020 guard is missing one "
            "of its required conditions."
        )

    guards_checked += 1


if guards_checked == 0:
    raise RuntimeError(
        "No generalized 2020 guards were found."
    )

print(
    "Generalized 2020 guards: PASS",
    guards_checked,
)


print()
print("CHENNAI_STRUCTURAL_AUDIT_SUCCESS")
PY


# ============================================================
# LAUNCH-STATE AUDIT
# ============================================================

if [ -e "$ROOT/batch/city.pid" ]; then
    echo "ERROR: Chennai city.pid already exists."
    exit 30
fi

if [ -e "$ROOT/batch/city.log" ]; then
    echo "ERROR: Chennai city.log already exists."
    exit 31
fi

echo "No stale Chennai launch state: PASS"


# ============================================================
# CLEAN TEMPORARY COMPILATION OUTPUT
# ============================================================

rm -rf "$COMPILE_ROOT"


echo
echo "CHENNAI_STATIC_AUDIT_SUCCESS"
'''


# Replace a literal placeholder instead of using an f-string.

cmd = remote_audit.replace(
    "__CHENNAI_ROOT__",
    root,
)


audit_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5 RESULT")
print("=" * 80)

print(
    "Return code:",
    audit_result.returncode,
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("Chennai source files modified: NO")
print("CELL 5: SUCCESS")

CHENNAI STATIC AND LEAKAGE AUDIT
Required Chennai files: PASS
Chennai Python compilation: PASS
Chennai configuration: PASS
Required production functions: PASS
Expected 4,900-cell constant: PASS {'EXPECTED_ROWS': 4900}
Driver template checksums: PASS
Workspace manifest checksums: PASS
Chennai paths and extent in code: PASS
Hyderabad leakage audit: PASS
Generalized template logic: PASS
Generalized 2020 guards: PASS 3

CHENNAI_STRUCTURAL_AUDIT_SUCCESS
No stale Chennai launch state: PASS

CHENNAI_STATIC_AUDIT_SUCCESS

REMOTE STDERR:


CELL 5 RESULT
Return code: 0
Spark launched: NO
HDFS modified: NO
Chennai source files modified: NO
CELL 5: SUCCESS


In [9]:
# ============================================================
# CELL 6A: SET CHENNAI SELECTED-TILE EXPECTATION TO 12
#
# Evidence from the Chennai 1975 production run:
#   Actual selected tiles: 12
#
# Updates only:
#   - Chennai population template
#   - Chennai built-surface template
#   - Chennai driver template checksums
#   - Chennai workspace manifest checksums
#
# Removes generated Chennai programs so they are regenerated.
#
# Does not:
#   - modify the frozen baseline
#   - modify Hyderabad, Pune, or Bengaluru
#   - modify HDFS
#   - launch Spark
# ============================================================

import base64
import hashlib
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


remote_patcher = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import sys
    import time


    ROOT = Path(sys.argv[1])

    DRIVER = (
        ROOT
        / "batch"
        / "run_city_epochs.py"
    )

    MANIFEST = (
        ROOT
        / "chennai_workspace_manifest.json"
    )

    POPULATION_TEMPLATE = (
        ROOT
        / "templates"
        / "population_epoch_template.py"
    )

    SMOD_TEMPLATE = (
        ROOT
        / "templates"
        / "smod_epoch_template.py"
    )

    BUILT_TEMPLATE = (
        ROOT
        / "templates"
        / "built_surface_epoch_template.py"
    )

    PROGRAMS_ROOT = (
        ROOT
        / "programs"
    )


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    for required_path in [
        DRIVER,
        MANIFEST,
        POPULATION_TEMPLATE,
        SMOD_TEMPLATE,
        BUILT_TEMPLATE,
    ]:
        if not required_path.is_file():
            raise FileNotFoundError(
                required_path
            )


    # --------------------------------------------------------
    # Patch the two raster templates that enforce tile count.
    # --------------------------------------------------------

    template_actions = {}

    for template_path in [
        POPULATION_TEMPLATE,
        BUILT_TEMPLATE,
    ]:
        source = template_path.read_text(
            encoding="utf-8"
        )

        old_assignment = (
            "EXPECTED_SELECTED_TILES = 16"
        )

        new_assignment = (
            "EXPECTED_SELECTED_TILES = 12"
        )

        old_count = source.count(
            old_assignment
        )

        new_count = source.count(
            new_assignment
        )

        if old_count == 1:
            patched_source = source.replace(
                old_assignment,
                new_assignment,
                1,
            )

            action = "changed_16_to_12"

        elif old_count == 0 and new_count == 1:
            patched_source = source
            action = "already_12"

        else:
            raise RuntimeError(
                "Unexpected tile-count structure in "
                + str(template_path)
                + ": old_count="
                + str(old_count)
                + ", new_count="
                + str(new_count)
            )

        compile(
            patched_source,
            str(template_path),
            "exec",
        )

        template_path.write_text(
            patched_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(template_path),
            doraise=True,
        )

        template_actions[
            template_path.name
        ] = action


    # --------------------------------------------------------
    # Refresh actual template checksums.
    # --------------------------------------------------------

    template_checksums = {
        "population": sha256_file(
            POPULATION_TEMPLATE
        ),
        "smod": sha256_file(
            SMOD_TEMPLATE
        ),
        "built_surface": sha256_file(
            BUILT_TEMPLATE
        ),
    }


    # --------------------------------------------------------
    # Update EXPECTED_TEMPLATE_SHA256 in the Chennai driver.
    # --------------------------------------------------------

    driver_source = DRIVER.read_text(
        encoding="utf-8"
    )

    driver_tree = ast.parse(
        driver_source
    )

    checksum_node = None

    for node in driver_tree.body:
        if not isinstance(
            node,
            ast.Assign,
        ):
            continue

        if any(
            isinstance(
                target,
                ast.Name,
            )
            and target.id
            == "EXPECTED_TEMPLATE_SHA256"
            for target
            in node.targets
        ):
            checksum_node = node
            break

    if checksum_node is None:
        raise RuntimeError(
            "EXPECTED_TEMPLATE_SHA256 "
            "was not found in the driver."
        )

    driver_lines = driver_source.splitlines(
        keepends=True
    )

    checksum_assignment = (
        "EXPECTED_TEMPLATE_SHA256 = "
        + repr(template_checksums)
        + chr(10)
    )

    driver_lines[
        checksum_node.lineno - 1:
        checksum_node.end_lineno
    ] = [
        checksum_assignment
    ]

    patched_driver_source = "".join(
        driver_lines
    )

    compile(
        patched_driver_source,
        str(DRIVER),
        "exec",
    )

    DRIVER.write_text(
        patched_driver_source,
        encoding="utf-8",
    )

    py_compile.compile(
        str(DRIVER),
        doraise=True,
    )


    # --------------------------------------------------------
    # Verify the saved driver checksum dictionary.
    # --------------------------------------------------------

    saved_tree = ast.parse(
        DRIVER.read_text(
            encoding="utf-8"
        )
    )

    saved_checksums = None

    for node in saved_tree.body:
        if not isinstance(
            node,
            ast.Assign,
        ):
            continue

        if any(
            isinstance(
                target,
                ast.Name,
            )
            and target.id
            == "EXPECTED_TEMPLATE_SHA256"
            for target
            in node.targets
        ):
            saved_checksums = ast.literal_eval(
                node.value
            )

            break

    if saved_checksums != template_checksums:
        raise RuntimeError(
            "Saved driver template checksums "
            "do not match the Chennai templates."
        )


    # --------------------------------------------------------
    # Refresh the Chennai workspace manifest.
    # --------------------------------------------------------

    workspace_manifest = json.loads(
        MANIFEST.read_text(
            encoding="utf-8"
        )
    )

    workspace_manifest[
        "driver_sha256"
    ] = sha256_file(
        DRIVER
    )

    workspace_manifest[
        "template_sha256"
    ] = template_checksums

    workspace_manifest[
        "expected_selected_tiles"
    ] = {
        "population": 12,
        "built_surface": 12,
    }

    workspace_manifest[
        "tile_expectation_updated_at_utc"
    ] = time.strftime(
        "%Y-%m-%dT%H:%M:%SZ",
        time.gmtime(),
    )

    MANIFEST.write_text(
        json.dumps(
            workspace_manifest,
            indent=2,
            sort_keys=True,
        )
        + chr(10),
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Remove generated programs from the failed launch.
    #
    # The production driver will regenerate them from the
    # corrected templates.
    # --------------------------------------------------------

    programs_removed = False

    if PROGRAMS_ROOT.exists():
        shutil.rmtree(
            PROGRAMS_ROOT
        )

        programs_removed = True

    PROGRAMS_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )


    # --------------------------------------------------------
    # Remove bytecode generated during compilation.
    # --------------------------------------------------------

    for pycache in ROOT.rglob(
        "__pycache__"
    ):
        shutil.rmtree(
            pycache
        )


    # --------------------------------------------------------
    # Final verification.
    # --------------------------------------------------------

    population_source = (
        POPULATION_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    built_source = (
        BUILT_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    if (
        population_source.count(
            "EXPECTED_SELECTED_TILES = 12"
        )
        != 1
    ):
        raise RuntimeError(
            "Population tile expectation "
            "was not saved as 12."
        )

    if (
        built_source.count(
            "EXPECTED_SELECTED_TILES = 12"
        )
        != 1
    ):
        raise RuntimeError(
            "Built-surface tile expectation "
            "was not saved as 12."
        )


    print("=" * 80)
    print("CHENNAI TILE EXPECTATION PATCH")
    print("=" * 80)

    print(
        "Population template:",
        template_actions[
            POPULATION_TEMPLATE.name
        ],
    )

    print(
        "Built-surface template:",
        template_actions[
            BUILT_TEMPLATE.name
        ],
    )

    print(
        "Generated programs removed:",
        programs_removed,
    )

    print()
    print("Population selected tiles: 12")
    print("Built-surface selected tiles: 12")
    print("SMOD template modified: NO")
    print("Driver checksums refreshed: PASS")
    print("Workspace manifest refreshed: PASS")
    print("Driver compilation: PASS")
    print("Generated programs reset: PASS")
    print("HDFS modified: NO")
    print("Spark launched: NO")
    print("CHENNAI TILE EXPECTATION PATCH: SUCCESS")
    '''
).strip() + "\n"


compile(
    remote_patcher,
    "patch_chennai_tile_expectation.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print(
    "LOCAL CHENNAI TILE PATCHER COMPILE: PASS"
)

print(
    "Expected patcher SHA-256:",
    patcher_sha256,
)


encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/patch_chennai_tile_expectation.py"
EXPECTED_SHA256="{patcher_sha256}"
ROOT="{root}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}" "${{ROOT}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 6A RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print("HDFS modified: NO")
print("Spark launched: NO")
print("CELL 6A: SUCCESS")

LOCAL CHENNAI TILE PATCHER COMPILE: PASS
Expected patcher SHA-256: 65454528493784a93648678a811477e65c2755fe03acca3691ba514656cc1c79
Remote patcher SHA-256: 65454528493784a93648678a811477e65c2755fe03acca3691ba514656cc1c79
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
CHENNAI TILE EXPECTATION PATCH
Population template: changed_16_to_12
Built-surface template: changed_16_to_12
Generated programs removed: True

Population selected tiles: 12
Built-surface selected tiles: 12
SMOD template modified: NO
Driver checksums refreshed: PASS
Workspace manifest refreshed: PASS
Driver compilation: PASS
Generated programs reset: PASS
HDFS modified: NO
Spark launched: NO
CHENNAI TILE EXPECTATION PATCH: SUCCESS

REMOTE STDERR:


CELL 6A RESULT
Return code: 0
HDFS modified: NO
Spark launched: NO
CELL 6A: SUCCESS


In [14]:
# ============================================================
# CELL 6B: INSPECT CHENNAI COASTAL-GRID CODE PATHS
#
# Completely read-only.
#
# Inspects the exact current Chennai:
#   - population template
#   - SMOD template
#   - built-surface template
#   - generated 1975 programs
#   - driver enrichment generator
#
# Does not:
#   - modify source files
#   - modify generated programs
#   - launch Spark
#   - modify HDFS
# ============================================================

import base64
import hashlib
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


remote_inspector = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import sys


    ROOT = Path(sys.argv[1])

    paths = {
        "driver": (
            ROOT
            / "batch"
            / "run_city_epochs.py"
        ),
        "population_template": (
            ROOT
            / "templates"
            / "population_epoch_template.py"
        ),
        "smod_template": (
            ROOT
            / "templates"
            / "smod_epoch_template.py"
        ),
        "built_template": (
            ROOT
            / "templates"
            / "built_surface_epoch_template.py"
        ),
        "population_1975": (
            ROOT
            / "programs"
            / "epoch_1975"
            / "population_1975.py"
        ),
        "smod_1975": (
            ROOT
            / "programs"
            / "epoch_1975"
            / "smod_1975.py"
        ),
        "built_1975": (
            ROOT
            / "programs"
            / "epoch_1975"
            / "built_surface_1975.py"
        ),
        "enriched_1975": (
            ROOT
            / "programs"
            / "epoch_1975"
            / "enrich_1975.py"
        ),
    }


    for name, path in paths.items():
        if name in {
            "driver",
            "population_template",
            "smod_template",
            "built_template",
        }:
            if not path.is_file():
                raise FileNotFoundError(
                    path
                )


    needles = [
        "Unexpected 2020 grid-cell count",
        "EXPECTED_ROWS",
        "EXPECTED_COMPLETE",
        "EXPECTED_PARTIAL",
        "EXPECTED_SELECTED_TILES",
        "grid_cell_count",
        "complete_cells",
        "partial_cells",
        "valid_100m_pixels",
        "valid_coverage_fraction",
        "population_total",
        "built_surface_total",
        "smod_class",
        "grid_cell_id",
        "groupBy(",
        ".groupBy(",
        ".join(",
        "join(",
        "left",
        "outer",
        "crossJoin",
        "range(",
        "sequence(",
        "grid_1km_col",
        "grid_1km_row",
        "write",
        ".parquet(",
    ]


    def print_context(
        path,
        needle,
        before=20,
        after=35,
    ):
        lines = path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

        matches = [
            index
            for index, line
            in enumerate(lines)
            if needle in line
        ]

        if not matches:
            return

        print()
        print(
            "NEEDLE:",
            repr(needle),
        )

        print(
            "MATCH LINES:",
            [
                index + 1
                for index in matches
            ],
        )

        for index in matches:
            start = max(
                0,
                index - before,
            )

            end = min(
                len(lines),
                index + after + 1,
            )

            print(
                "--- "
                + path.name
                + " lines "
                + str(start + 1)
                + "-"
                + str(end)
                + " ---"
            )

            for current in range(
                start,
                end,
            ):
                print(
                    str(current + 1).rjust(5)
                    + ": "
                    + lines[current]
                )


    for name, path in paths.items():
        print()
        print("=" * 80)
        print(name.upper())
        print("=" * 80)
        print("Path:", path)
        print("Exists:", path.is_file())

        if not path.is_file():
            continue

        source = path.read_text(
            encoding="utf-8",
            errors="replace",
        )

        print(
            "Bytes:",
            path.stat().st_size,
        )

        try:
            ast.parse(source)

            print(
                "Python AST:",
                "PASS",
            )

        except SyntaxError as error:
            print(
                "Python AST:",
                "FAIL",
                repr(error),
            )

        for needle in needles:
            print_context(
                path,
                needle,
            )


    print()
    print("=" * 80)
    print("COASTAL-GRID INSPECTION COMPLETE")
    print("=" * 80)

    print("Files modified: NO")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print("CHENNAI_COASTAL_GRID_INSPECTION_SUCCESS")
    '''
).strip() + "\n"


compile(
    remote_inspector,
    "inspect_chennai_coastal_grid.py",
    "exec",
)

inspector_sha256 = hashlib.sha256(
    remote_inspector.encode("utf-8")
).hexdigest()

print(
    "LOCAL COASTAL-GRID INSPECTOR COMPILE: PASS"
)

print(
    "Expected inspector SHA-256:",
    inspector_sha256,
)


encoded_inspector = base64.b64encode(
    remote_inspector.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

INSPECTOR="/tmp/inspect_chennai_coastal_grid.py"
EXPECTED_SHA256="{inspector_sha256}"
ROOT="{root}"

echo "{encoded_inspector}" \
| base64 -d \
> "${{INSPECTOR}}"

ACTUAL_SHA256="$(
    sha256sum "${{INSPECTOR}}" \
    | awk '{{print $1}}'
)"

echo "Remote inspector SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Inspector checksum mismatch."
    exit 20
fi

echo "REMOTE INSPECTOR CHECKSUM: PASS"

python3 -m py_compile "${{INSPECTOR}}"

echo "REMOTE INSPECTOR COMPILE: PASS"

python3 "${{INSPECTOR}}" "${{ROOT}}"
"""


inspection_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 6B RESULT")
print("=" * 80)

print(
    "Return code:",
    inspection_result.returncode,
)

print("Files modified: NO")
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 6B: SUCCESS")

Streaming output truncated to the last 5000 lines.
  688:             COUNT(*) * {SOURCE_PIXEL_AREA_M2}
  689:                 AS observed_grid_area_m2,
  690: 
  691:             COUNT(*) / 100.0
  692:                 AS built_coverage_fraction,
  693: 
  694:             SUM(
  695:                 built_surface_total_m2_100m
  696:             ) AS built_surface_total_m2,
  697: 
  698:             SUM(
  699:                 built_surface_nres_m2_100m
  700:             ) AS built_surface_nres_m2,
  701: 
  702:             SUM(
  703:                 built_surface_residential_m2_100m
  704:             ) AS built_surface_residential_m2,
--- built_surface_epoch_template.py lines 710-765 ---
  710:                     ELSE 0
  711:                 END
  712:             ) AS built_100m_pixels,
  713: 
  714:             SUM(
  715:                 CASE
  716:                     WHEN built_surface_nres_m2_100m > 0.0
  717:                     THEN 1
  718:                     ELSE 

In [15]:
# ============================================================
# CELL 6C: TARGETED COASTAL-GRID SOURCE EXTRACTION
#
# Read-only.
# Does not launch Spark or modify HDFS/source files.
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )

cmd = r'''
set -e

ROOT="__CHENNAI_ROOT__"

echo "============================================================"
echo "POPULATION TEMPLATE: AGGREGATION THROUGH VALIDATION"
echo "============================================================"

sed -n '530,790p' \
    "$ROOT/templates/population_epoch_template.py"

echo
echo "============================================================"
echo "SMOD TEMPLATE: AGGREGATION THROUGH WRITE"
echo "============================================================"

grep -n \
    -E \
    'smod_1km_df|GROUP BY|VALIDATE|GRID_YEAR_REFERENCE|population|join|WRITE HDFS|parquet' \
    "$ROOT/templates/smod_epoch_template.py" \
    | head -n 120

echo
echo "----- SMOD SOURCE LINES 500-1050 -----"

sed -n '500,1050p' \
    "$ROOT/templates/smod_epoch_template.py"

echo
echo "============================================================"
echo "BUILT TEMPLATE: AGGREGATION THROUGH ALIGNMENT"
echo "============================================================"

sed -n '620,1040p' \
    "$ROOT/templates/built_surface_epoch_template.py"

echo
echo "============================================================"
echo "DRIVER: ENRICHMENT GENERATOR AND VALIDATOR"
echo "============================================================"

grep -n \
    -E \
    'generate_enrich|enrich_|ENRICHED|population_hdfs|smod_hdfs|built_hdfs|join|EXPECTED_ROWS' \
    "$ROOT/batch/run_city_epochs.py"

echo
echo "----- DRIVER FULL DEFINITIONS AREA -----"

sed -n '1,260p' \
    "$ROOT/batch/run_city_epochs.py"

echo
echo "============================================================"
echo "TARGETED EXTRACTION COMPLETE"
echo "============================================================"

echo "Files modified: NO"
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "CHENNAI_TARGETED_EXTRACTION_SUCCESS"
'''

cmd = cmd.replace(
    "__CHENNAI_ROOT__",
    root,
)

result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 6C RESULT")
print("=" * 80)

print("Return code:", result.returncode)
print("Files modified: NO")
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 6C: SUCCESS")

POPULATION TEMPLATE: AGGREGATION THROUGH VALIDATION
            pixel_center_x >= {PILOT_MIN_X}
            AND pixel_center_x < {PILOT_MAX_X}
            AND pixel_center_y >= {PILOT_MIN_Y}
            AND pixel_center_y < {PILOT_MAX_Y}
            AND is_valid_population
        """
    )

    valid_pixels_df.createOrReplaceTempView(
        "valid_population_pixels"
    )

    valid_pixel_count = (
        valid_pixels_df.count()
    )

    if valid_pixel_count <= 0:
        raise RuntimeError(
            "No valid 2020 pilot pixels were found."
        )

    print(
        "Valid 2020 pilot pixels:",
        valid_pixel_count,
    )


    # ====================================================
    # AGGREGATE 100 M PIXELS TO 1 KM
    # ====================================================

    population_1km_df = spark.sql(
        f"""
        SELECT
            '{CITY}' AS city,
            '{PRODUCT}' AS product,
            CAST({EPOCH} AS INT) AS epoch,

            grid_1km_c

In [16]:
# ============================================================
# CELL 6D: EXPORT EXACT CHENNAI PIPELINE SOURCES
#
# Read-only:
#   - Copies the current Chennai driver and templates
#   - Creates a compressed archive on EC2
#   - Downloads the archive into Colab
#   - Does not alter source files
#   - Does not launch Spark
#   - Does not modify HDFS
# ============================================================

from pathlib import Path
import subprocess


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


remote_archive = (
    "/tmp/chennai_coastal_grid_sources.tar.gz"
)

local_archive = Path(
    "/content/chennai_coastal_grid_sources.tar.gz"
)


cmd = r'''
set -e

ROOT="__CHENNAI_ROOT__"
ARCHIVE="/tmp/chennai_coastal_grid_sources.tar.gz"
STAGING="/tmp/chennai_coastal_grid_sources"

rm -rf "$STAGING"
rm -f "$ARCHIVE"

mkdir -p \
    "$STAGING/batch" \
    "$STAGING/templates"

cp \
    "$ROOT/batch/run_city_epochs.py" \
    "$STAGING/batch/run_city_epochs.py"

cp \
    "$ROOT/templates/population_epoch_template.py" \
    "$STAGING/templates/population_epoch_template.py"

cp \
    "$ROOT/templates/smod_epoch_template.py" \
    "$STAGING/templates/smod_epoch_template.py"

cp \
    "$ROOT/templates/built_surface_epoch_template.py" \
    "$STAGING/templates/built_surface_epoch_template.py"

cp \
    "$ROOT/city_config.json" \
    "$STAGING/city_config.json"

cp \
    "$ROOT/chennai_workspace_manifest.json" \
    "$STAGING/chennai_workspace_manifest.json"

(
    cd "$STAGING"

    sha256sum \
        batch/run_city_epochs.py \
        templates/population_epoch_template.py \
        templates/smod_epoch_template.py \
        templates/built_surface_epoch_template.py \
        city_config.json \
        chennai_workspace_manifest.json \
        > SHA256SUMS
)

tar \
    -czf "$ARCHIVE" \
    -C "/tmp" \
    "chennai_coastal_grid_sources"

test -s "$ARCHIVE"

echo "============================================================"
echo "CHENNAI SOURCE EXPORT"
echo "============================================================"

echo "Archive: $ARCHIVE"
echo "Archive bytes: $(stat -c %s "$ARCHIVE")"
echo "Archive SHA-256: $(sha256sum "$ARCHIVE" | awk '{print $1}')"

echo
echo "Contents:"

tar -tzf "$ARCHIVE"

echo
echo "Source files modified: NO"
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "CHENNAI_SOURCE_EXPORT_SUCCESS"
'''


cmd = cmd.replace(
    "__CHENNAI_ROOT__",
    root,
)


run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


# remote_exec.sh already contains the EC2 connection details.
# Use its SSH configuration to copy the archive back to Colab.

remote_exec_path = Path(
    "/content/drive/MyDrive/remote_exec.sh"
)

if not remote_exec_path.is_file():
    raise FileNotFoundError(
        remote_exec_path
    )


# Inspect the helper for its SSH destination and options.
helper_text = remote_exec_path.read_text(
    encoding="utf-8"
)

print()
print(
    "Remote archive created successfully:",
    remote_archive,
)

print(
    "Download it from EC2 using your existing "
    "SSH key, or run the SCP command used by "
    "remote_exec.sh."
)

print()
print(
    "Archive name:",
    local_archive.name,
)

print("CELL 6D: REMOTE EXPORT SUCCESS")

CHENNAI SOURCE EXPORT
Archive: /tmp/chennai_coastal_grid_sources.tar.gz
Archive bytes: 17717
Archive SHA-256: 0475a340d19bfa411e75d7136b60cc1093395339c9481f5e91c3f14ae47c709b

Contents:
chennai_coastal_grid_sources/
chennai_coastal_grid_sources/SHA256SUMS
chennai_coastal_grid_sources/chennai_workspace_manifest.json
chennai_coastal_grid_sources/city_config.json
chennai_coastal_grid_sources/templates/
chennai_coastal_grid_sources/templates/built_surface_epoch_template.py
chennai_coastal_grid_sources/templates/smod_epoch_template.py
chennai_coastal_grid_sources/templates/population_epoch_template.py
chennai_coastal_grid_sources/batch/
chennai_coastal_grid_sources/batch/run_city_epochs.py

Source files modified: NO
Spark launched: NO
HDFS modified: NO
CHENNAI_SOURCE_EXPORT_SUCCESS

REMOTE STDERR:


Remote archive created successfully: /tmp/chennai_coastal_grid_sources.tar.gz
Download it from EC2 using your existing SSH key, or run the SCP command used by remote_exec.sh.

Archive name: chen

In [18]:
# ============================================================
# CELL 6E-DOWNLOAD: TRANSFER ARCHIVE THROUGH run_remote()
#
# EC2 source:
#   /tmp/chennai_coastal_grid_sources.tar.gz
#
# Colab destination:
#   /content/chennai_coastal_grid_sources.tar.gz
#
# Read-only toward EC2 and HDFS.
# ============================================================

from pathlib import Path
import base64
import hashlib


REMOTE_ARCHIVE = (
    "/tmp/chennai_coastal_grid_sources.tar.gz"
)

LOCAL_ARCHIVE = Path(
    "/content/chennai_coastal_grid_sources.tar.gz"
)

EXPECTED_SHA256 = (
    "0475a340d19bfa411e75d7136b60cc1093395339c9481f5e91c3f14ae47c709b"
)


# ============================================================
# READ THE ARCHIVE ON EC2 AS BASE64
#
# No SCP parsing is required. The existing run_remote() helper
# continues to handle the SSH connection.
# ============================================================

remote_command = f"""
set -e

ARCHIVE="{REMOTE_ARCHIVE}"

if [ ! -f "${{ARCHIVE}}" ]; then
    echo "ERROR: Archive does not exist." >&2
    exit 20
fi

if [ ! -s "${{ARCHIVE}}" ]; then
    echo "ERROR: Archive is empty." >&2
    exit 21
fi

base64 -w 0 "${{ARCHIVE}}"
"""


transfer_result = run_remote(
    remote_command,
    timeout=300,
    show_output=False,
)


# ============================================================
# VALIDATE REMOTE COMMAND RESULT
# ============================================================

if transfer_result.returncode != 0:
    if transfer_result.stderr:
        print(transfer_result.stderr)

    raise RuntimeError(
        "Remote archive transfer failed with "
        "return code "
        + str(transfer_result.returncode)
    )


encoded_archive = (
    transfer_result.stdout.strip()
)


if not encoded_archive:
    raise RuntimeError(
        "The remote command returned no archive data."
    )


# ============================================================
# DECODE AND SAVE IN COLAB
# ============================================================

try:
    archive_bytes = base64.b64decode(
        encoded_archive,
        validate=True,
    )

except Exception as error:
    raise RuntimeError(
        "The transferred archive was not valid Base64."
    ) from error


LOCAL_ARCHIVE.write_bytes(
    archive_bytes
)


# ============================================================
# VERIFY SIZE AND SHA-256
# ============================================================

if not LOCAL_ARCHIVE.is_file():
    raise FileNotFoundError(
        LOCAL_ARCHIVE
    )


actual_size = (
    LOCAL_ARCHIVE.stat().st_size
)

actual_sha256 = hashlib.sha256(
    LOCAL_ARCHIVE.read_bytes()
).hexdigest()


if actual_size != 17717:
    raise RuntimeError(
        "Downloaded archive size mismatch.\n"
        "Expected: 17717 bytes\n"
        "Actual:   "
        + str(actual_size)
        + " bytes"
    )


if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "Downloaded archive checksum mismatch.\n"
        "Expected: "
        + EXPECTED_SHA256
        + "\nActual:   "
        + actual_sha256
    )


print("=" * 80)
print("CHENNAI SOURCE ARCHIVE TRANSFER")
print("=" * 80)

print(
    "Remote source:",
    REMOTE_ARCHIVE,
)

print(
    "Colab file:",
    LOCAL_ARCHIVE,
)

print(
    "Archive bytes:",
    actual_size,
)

print(
    "Archive SHA-256:",
    actual_sha256,
)

print()
print("Archive size verification: PASS")
print("Archive checksum verification: PASS")
print("EC2 source files modified: NO")
print("HDFS modified: NO")
print("CELL 6E-DOWNLOAD: SUCCESS")

CHENNAI SOURCE ARCHIVE TRANSFER
Remote source: /tmp/chennai_coastal_grid_sources.tar.gz
Colab file: /content/chennai_coastal_grid_sources.tar.gz
Archive bytes: 17717
Archive SHA-256: 0475a340d19bfa411e75d7136b60cc1093395339c9481f5e91c3f14ae47c709b

Archive size verification: PASS
Archive checksum verification: PASS
EC2 source files modified: NO
HDFS modified: NO
CELL 6E-DOWNLOAD: SUCCESS


In [21]:
# ============================================================
# CELL 6G: CHENNAI COASTAL CANONICAL-GRID CORRECTION
#
# Design:
#   - Retains all 4,900 canonical 1 km cells
#   - Keeps observed population zeros unchanged
#   - Fills cells without raster coverage with:
#       population_total = 0.0
#       smod_class = 0
#       built-surface values = 0.0
#   - Adds explicit coverage indicators
#
# Also:
#   - refreshes driver template checksums
#   - refreshes the Chennai workspace manifest
#   - removes stale generated programs
#   - preserves the failed production log
#   - removes only a stopped/stale PID file
#   - rolls back source files if any patch check fails
#
# Does not:
#   - modify the frozen baseline
#   - modify HDFS
#   - launch Spark
# ============================================================

import base64
import hashlib
import json
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


# ============================================================
# PATCH FRAGMENT 1: POPULATION CANONICAL GRID
# ============================================================

population_patch = r'''
    # ====================================================
    # COASTAL CANONICAL-GRID COMPLETION
    #
    # The raster aggregation contains only cells with valid
    # pixels. Create the complete 70 x 70 canonical grid and
    # left join the raster aggregation onto it.
    # ====================================================

    population_raster_aggregate_df = (
        population_1km_df
    )

    canonical_columns_df = (
        spark.range(
            int(
                PILOT_MIN_X
                / TARGET_RESOLUTION
            ),
            int(
                PILOT_MAX_X
                / TARGET_RESOLUTION
            ),
        )
        .select(
            F.col("id")
            .cast("long")
            .alias("grid_1km_col")
        )
    )

    canonical_rows_df = (
        spark.range(
            int(
                PILOT_MIN_Y
                / TARGET_RESOLUTION
            ),
            int(
                PILOT_MAX_Y
                / TARGET_RESOLUTION
            ),
        )
        .select(
            F.col("id")
            .cast("long")
            .alias("grid_1km_row")
        )
    )

    canonical_population_grid_df = (
        canonical_columns_df
        .crossJoin(
            canonical_rows_df
        )
        .withColumn(
            "grid_cell_id",
            F.concat(
                F.lit("MOLL_1KM_"),
                F.col(
                    "grid_1km_col"
                ).cast("string"),
                F.lit("_"),
                F.col(
                    "grid_1km_row"
                ).cast("string"),
            ),
        )
        .withColumn(
            "grid_min_x",
            F.col("grid_1km_col")
            * F.lit(
                float(
                    TARGET_RESOLUTION
                )
            ),
        )
        .withColumn(
            "grid_max_x",
            (
                F.col("grid_1km_col")
                + F.lit(1)
            )
            * F.lit(
                float(
                    TARGET_RESOLUTION
                )
            ),
        )
        .withColumn(
            "grid_min_y",
            F.col("grid_1km_row")
            * F.lit(
                float(
                    TARGET_RESOLUTION
                )
            ),
        )
        .withColumn(
            "grid_max_y",
            (
                F.col("grid_1km_row")
                + F.lit(1)
            )
            * F.lit(
                float(
                    TARGET_RESOLUTION
                )
            ),
        )
    )

    population_1km_df = (
        canonical_population_grid_df.alias("g")
        .join(
            population_raster_aggregate_df.alias("a"),
            on="grid_cell_id",
            how="left",
        )
        .select(
            F.lit(CITY).alias("city"),
            F.lit(PRODUCT).alias("product"),
            F.lit(EPOCH).cast("int").alias("epoch"),

            F.col("g.grid_1km_col"),
            F.col("g.grid_1km_row"),
            F.col("g.grid_cell_id"),

            F.col("g.grid_min_x"),
            F.col("g.grid_max_x"),
            F.col("g.grid_min_y"),
            F.col("g.grid_max_y"),

            F.coalesce(
                F.col("a.valid_100m_pixels"),
                F.lit(0),
            )
            .cast("long")
            .alias("valid_100m_pixels"),

            F.coalesce(
                F.col(
                    "a.valid_coverage_fraction"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "valid_coverage_fraction"
            ),

            F.coalesce(
                F.col("a.population_total"),
                F.lit(0.0),
            )
            .cast("double")
            .alias("population_total"),

            F.col(
                "a.population_mean_100m"
            ).alias(
                "population_mean_100m"
            ),

            F.col(
                "a.population_min_100m"
            ).alias(
                "population_min_100m"
            ),

            F.col(
                "a.population_max_100m"
            ).alias(
                "population_max_100m"
            ),

            F.coalesce(
                F.col(
                    "a.zero_population_pixels"
                ),
                F.lit(0),
            )
            .cast("long")
            .alias(
                "zero_population_pixels"
            ),

            F.coalesce(
                F.col(
                    "a.positive_population_pixels"
                ),
                F.lit(0),
            )
            .cast("long")
            .alias(
                "positive_population_pixels"
            ),

            F.coalesce(
                F.col("a.source_path"),
                F.lit(
                    "no_valid_population_coverage"
                ),
            ).alias("source_path"),

            F.lit("100.0").alias(
                "input_resolution_m"
            ),

            F.lit("1000.0").alias(
                "output_resolution_m"
            ),

            F.current_timestamp().alias(
                "processed_at_utc"
            ),
        )
        .withColumn(
            "has_valid_population_coverage",
            F.col("valid_100m_pixels")
            > F.lit(0),
        )
    )
'''.strip("\n")


# ============================================================
# PATCH FRAGMENT 2: SMOD CANONICAL ALIGNMENT
# ============================================================

smod_patch = r'''
    # ====================================================
    # COASTAL CANONICAL-GRID COMPLETION
    #
    # Align SMOD to the complete population grid.
    # Class 0 represents no valid SMOD raster coverage.
    # ====================================================

    smod_raster_grid_df = (
        smod_grid_df
    )

    canonical_smod_reference_df = (
        spark.read
        .parquet(
            POPULATION_PATH
        )
        .select(
            "grid_cell_id",
            "epoch",
            "grid_1km_col",
            "grid_1km_row",
            "grid_min_x",
            "grid_max_x",
            "grid_min_y",
            "grid_max_y",
        )
    )

    smod_grid_df = (
        canonical_smod_reference_df.alias("p")
        .join(
            smod_raster_grid_df.alias("s"),
            on=[
                F.col("p.grid_cell_id")
                == F.col("s.grid_cell_id"),
                F.col("p.epoch")
                == F.col("s.epoch"),
            ],
            how="left",
        )
        .select(
            F.lit(CITY).alias("city"),
            F.lit(PRODUCT).alias("product"),

            F.col("p.epoch"),
            F.col("p.grid_1km_col"),
            F.col("p.grid_1km_row"),
            F.col("p.grid_cell_id"),

            F.coalesce(
                F.col("s.smod_class"),
                F.lit(0),
            )
            .cast("int")
            .alias("smod_class"),

            F.col("p.grid_min_x"),
            F.col("p.grid_max_x"),
            F.col("p.grid_min_y"),
            F.col("p.grid_max_y"),

            F.coalesce(
                F.col("s.source_path"),
                F.lit(
                    "no_valid_smod_coverage"
                ),
            ).alias("source_path"),

            F.current_timestamp().alias(
                "processed_at_utc"
            ),

            F.col("s.smod_class")
            .isNotNull()
            .alias(
                "has_valid_smod_coverage"
            ),
        )
    )
'''.strip("\n")


# ============================================================
# PATCH FRAGMENT 3: BUILT-SURFACE CANONICAL GRID
# ============================================================

built_patch = r'''
    # ====================================================
    # COASTAL CANONICAL-GRID COMPLETION
    #
    # Align built-surface aggregates to the complete
    # population reference grid.
    # ====================================================

    built_raster_aggregate_df = (
        built_1km_df
    )

    canonical_built_reference_df = (
        spark.read
        .parquet(
            GRID_YEAR_REFERENCE_PATH
        )
        .select(
            "grid_cell_id",
            "epoch",
            "grid_1km_col",
            "grid_1km_row",
            "grid_min_x",
            "grid_max_x",
            "grid_min_y",
            "grid_max_y",
        )
    )

    built_1km_df = (
        canonical_built_reference_df.alias("p")
        .join(
            built_raster_aggregate_df.alias("b"),
            on=[
                F.col("p.grid_cell_id")
                == F.col("b.grid_cell_id"),
                F.col("p.epoch")
                == F.col("b.epoch"),
            ],
            how="left",
        )
        .select(
            F.lit(CITY).alias("city"),
            F.lit(PRODUCT).alias("product"),

            F.col("p.epoch"),
            F.col("p.grid_1km_col"),
            F.col("p.grid_1km_row"),
            F.col("p.grid_cell_id"),

            F.col("p.grid_min_x"),
            F.col("p.grid_max_x"),
            F.col("p.grid_min_y"),
            F.col("p.grid_max_y"),

            F.coalesce(
                F.col("b.valid_100m_pixels"),
                F.lit(0),
            )
            .cast("long")
            .alias("valid_100m_pixels"),

            F.coalesce(
                F.col(
                    "b.observed_grid_area_m2"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "observed_grid_area_m2"
            ),

            F.coalesce(
                F.col(
                    "b.built_coverage_fraction"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "built_coverage_fraction"
            ),

            F.coalesce(
                F.col(
                    "b.built_surface_total_m2"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "built_surface_total_m2"
            ),

            F.coalesce(
                F.col(
                    "b.built_surface_nres_m2"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "built_surface_nres_m2"
            ),

            F.coalesce(
                F.col(
                    "b.built_surface_residential_m2"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "built_surface_residential_m2"
            ),

            F.coalesce(
                F.col("b.built_100m_pixels"),
                F.lit(0),
            )
            .cast("long")
            .alias("built_100m_pixels"),

            F.coalesce(
                F.col("b.nres_100m_pixels"),
                F.lit(0),
            )
            .cast("long")
            .alias("nres_100m_pixels"),

            F.coalesce(
                F.col("b.source_path"),
                F.lit(
                    "no_valid_built_coverage"
                ),
            ).alias("source_path"),

            F.current_timestamp().alias(
                "processed_at_utc"
            ),
        )
        .withColumn(
            "has_valid_built_coverage",
            F.col("valid_100m_pixels")
            > F.lit(0),
        )
    )
'''.strip("\n")


# ============================================================
# PATCH FRAGMENT 4: DRIVER WRAPPERS
# ============================================================

driver_wrapper = r'''
_original_create_enrichment_program = (
    create_enrichment_program
)


def create_enrichment_program(
    epoch,
    epoch_dir,
):
    output_path = (
        _original_create_enrichment_program(
            epoch,
            epoch_dir,
        )
    )

    source = output_path.read_text(
        encoding="utf-8"
    )

    old_start = (
        "    if complete_count "
        "!= EXPECTED_COMPLETE:"
    )

    write_marker = (
        "\n\n\n    (\n"
        "        enriched_df\n"
        "        .repartition(4)"
    )

    start_index = source.find(
        old_start
    )

    end_index = source.find(
        write_marker,
        start_index,
    )

    if (
        start_index < 0
        or end_index < 0
    ):
        raise RuntimeError(
            "Could not locate the enrichment "
            "complete/partial validation block."
        )

    replacement = (
        "    if (\n"
        "        complete_count\n"
        "        + partial_count\n"
        "        != EXPECTED_ROWS\n"
        "    ):\n"
        "        raise RuntimeError(\n"
        '            "Complete plus partial cell "\n'
        '            "count does not equal the "\n'
        '            "canonical grid."\n'
        "        )\n"
    )

    source = (
        source[:start_index]
        + replacement
        + source[end_index:]
    )

    old_quality = (
        'F.lit("partial_pilot_boundary")'
    )

    new_quality = (
        "F.when(\n"
        '                    '
        'F.col("valid_100m_pixels") == 0,\n'
        '                    '
        'F.lit("no_coverage"),\n'
        "                ).otherwise(\n"
        '                    '
        'F.lit("partial_pilot_boundary")\n'
        "                )"
    )

    if source.count(old_quality) != 1:
        raise RuntimeError(
            "Could not locate the population "
            "quality-flag fallback."
        )

    source = source.replace(
        old_quality,
        new_quality,
        1,
    )

    compile(
        source,
        str(output_path),
        "exec",
    )

    output_path.write_text(
        source,
        encoding="utf-8",
    )

    return output_path
'''.strip("\n")


# ============================================================
# ENCODE PATCH PAYLOAD
# ============================================================

patch_payload = {
    "population_patch": population_patch,
    "smod_patch": smod_patch,
    "built_patch": built_patch,
    "driver_wrapper": driver_wrapper,
}

payload_json = json.dumps(
    patch_payload,
    sort_keys=True,
)

payload_b64 = base64.b64encode(
    payload_json.encode("utf-8")
).decode("ascii")

payload_sha256 = hashlib.sha256(
    payload_json.encode("utf-8")
).hexdigest()


# ============================================================
# REMOTE PATCHER
#
# Patch fragments are loaded from JSON. There are no nested
# triple-quoted patch strings in this remote source.
# ============================================================

remote_patcher = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import sys
    import time


    ROOT = Path(sys.argv[1])
    PAYLOAD_PATH = Path(sys.argv[2])


    DRIVER = (
        ROOT
        / "batch"
        / "run_city_epochs.py"
    )

    POPULATION_TEMPLATE = (
        ROOT
        / "templates"
        / "population_epoch_template.py"
    )

    SMOD_TEMPLATE = (
        ROOT
        / "templates"
        / "smod_epoch_template.py"
    )

    BUILT_TEMPLATE = (
        ROOT
        / "templates"
        / "built_surface_epoch_template.py"
    )

    WORKSPACE_MANIFEST = (
        ROOT
        / "chennai_workspace_manifest.json"
    )

    PROGRAMS_ROOT = (
        ROOT
        / "programs"
    )

    PID_FILE = (
        ROOT
        / "batch"
        / "city.pid"
    )

    LOG_FILE = (
        ROOT
        / "batch"
        / "city.log"
    )

    BACKUP_ROOT = (
        ROOT
        / "coastal_patch_backup"
    )

    PATCH_VERSION = (
        "chennai_coastal_canonical_grid_v1"
    )


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def require_once(
        source,
        marker,
        description,
    ):
        count = source.count(
            marker
        )

        if count != 1:
            raise RuntimeError(
                description
                + ": expected one occurrence, found "
                + str(count)
                + "\nMarker:\n"
                + marker
            )


    def insert_before_once(
        source,
        marker,
        insertion,
        description,
    ):
        require_once(
            source,
            marker,
            description,
        )

        return source.replace(
            marker,
            insertion
            + "\n\n"
            + marker,
            1,
        )


    required_files = [
        DRIVER,
        POPULATION_TEMPLATE,
        SMOD_TEMPLATE,
        BUILT_TEMPLATE,
        WORKSPACE_MANIFEST,
        PAYLOAD_PATH,
    ]

    for required_file in required_files:
        if not required_file.is_file():
            raise FileNotFoundError(
                required_file
            )


    payload = json.loads(
        PAYLOAD_PATH.read_text(
            encoding="utf-8"
        )
    )

    population_patch = payload[
        "population_patch"
    ]

    smod_patch = payload[
        "smod_patch"
    ]

    built_patch = payload[
        "built_patch"
    ]

    driver_wrapper = payload[
        "driver_wrapper"
    ]


    # ========================================================
    # REFUSE TO PATCH A RUNNING JOB
    # ========================================================

    if PID_FILE.is_file():
        recorded_pid = (
            PID_FILE.read_text(
                encoding="utf-8"
            ).strip()
        )

        if recorded_pid:
            process_path = (
                Path("/proc")
                / recorded_pid
            )

            if process_path.exists():
                raise RuntimeError(
                    "Chennai production is still running "
                    "with PID "
                    + recorded_pid
                )


    # ========================================================
    # CREATE TRANSACTIONAL BACKUP
    # ========================================================

    if BACKUP_ROOT.exists():
        shutil.rmtree(
            BACKUP_ROOT
        )

    (
        BACKUP_ROOT
        / "batch"
    ).mkdir(
        parents=True
    )

    (
        BACKUP_ROOT
        / "templates"
    ).mkdir()

    shutil.copy2(
        DRIVER,
        BACKUP_ROOT
        / "batch"
        / DRIVER.name,
    )

    shutil.copy2(
        POPULATION_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / POPULATION_TEMPLATE.name,
    )

    shutil.copy2(
        SMOD_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / SMOD_TEMPLATE.name,
    )

    shutil.copy2(
        BUILT_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / BUILT_TEMPLATE.name,
    )

    shutil.copy2(
        WORKSPACE_MANIFEST,
        BACKUP_ROOT
        / WORKSPACE_MANIFEST.name,
    )


    def restore_backup():
        shutil.copy2(
            BACKUP_ROOT
            / "batch"
            / DRIVER.name,
            DRIVER,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / POPULATION_TEMPLATE.name,
            POPULATION_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / SMOD_TEMPLATE.name,
            SMOD_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / BUILT_TEMPLATE.name,
            BUILT_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / WORKSPACE_MANIFEST.name,
            WORKSPACE_MANIFEST,
        )


    try:
        # ====================================================
        # PATCH POPULATION
        # ====================================================

        population_source = (
            POPULATION_TEMPLATE.read_text(
                encoding="utf-8"
            )
        )

        population_marker = (
            "    # ====================================================\n"
            "    # VALIDATE 1 KM OUTPUT\n"
            "    # ===================================================="
        )

        if (
            "COASTAL CANONICAL-GRID COMPLETION"
            not in population_source
        ):
            population_source = (
                insert_before_once(
                    population_source,
                    population_marker,
                    population_patch,
                    "Population coastal-grid insertion",
                )
            )

        old_pixel_check = (
            'F.col("valid_100m_pixels")\n'
            "                <= F.lit(0)"
        )

        new_pixel_check = (
            'F.col("valid_100m_pixels")\n'
            "                < F.lit(0)"
        )

        if old_pixel_check in population_source:
            population_source = (
                population_source.replace(
                    old_pixel_check,
                    new_pixel_check,
                    1,
                )
            )

        old_coverage_check = (
            'F.col(\n'
            '                    "valid_coverage_fraction"\n'
            "                )\n"
            "                <= F.lit(0.0)"
        )

        new_coverage_check = (
            'F.col(\n'
            '                    "valid_coverage_fraction"\n'
            "                )\n"
            "                < F.lit(0.0)"
        )

        if old_coverage_check in population_source:
            population_source = (
                population_source.replace(
                    old_coverage_check,
                    new_coverage_check,
                    1,
                )
            )

        compile(
            population_source,
            str(POPULATION_TEMPLATE),
            "exec",
        )

        POPULATION_TEMPLATE.write_text(
            population_source,
            encoding="utf-8",
        )


        # ====================================================
        # PATCH SMOD
        # ====================================================

        smod_source = (
            SMOD_TEMPLATE.read_text(
                encoding="utf-8"
            )
        )

        smod_marker = (
            "    # ====================================================\n"
            "    # VALIDATE THE 2020 SMOD GRID\n"
            "    # ===================================================="
        )

        if (
            "COASTAL CANONICAL-GRID COMPLETION"
            not in smod_source
        ):
            smod_source = insert_before_once(
                smod_source,
                smod_marker,
                smod_patch,
                "SMOD coastal-grid insertion",
            )

        compile(
            smod_source,
            str(SMOD_TEMPLATE),
            "exec",
        )

        SMOD_TEMPLATE.write_text(
            smod_source,
            encoding="utf-8",
        )


        # ====================================================
        # PATCH BUILT SURFACE
        # ====================================================

        built_source = (
            BUILT_TEMPLATE.read_text(
                encoding="utf-8"
            )
        )

        built_marker = (
            "    # ====================================================\n"
            "    # DERIVED FRACTIONS\n"
        )

        if (
            "COASTAL CANONICAL-GRID COMPLETION"
            not in built_source
        ):
            built_source = insert_before_once(
                built_source,
                built_marker,
                built_patch,
                "Built coastal-grid insertion",
            )

        compile(
            built_source,
            str(BUILT_TEMPLATE),
            "exec",
        )

        BUILT_TEMPLATE.write_text(
            built_source,
            encoding="utf-8",
        )


        # ====================================================
        # REFRESH TEMPLATE CHECKSUMS
        # ====================================================

        template_checksums = {
            "population": sha256_file(
                POPULATION_TEMPLATE
            ),
            "smod": sha256_file(
                SMOD_TEMPLATE
            ),
            "built_surface": sha256_file(
                BUILT_TEMPLATE
            ),
        }


        # ====================================================
        # PATCH DRIVER
        # ====================================================

        driver_source = DRIVER.read_text(
            encoding="utf-8"
        )

        driver_tree = ast.parse(
            driver_source
        )

        checksum_node = None

        for node in driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target in node.targets
            ):
                checksum_node = node
                break

        if checksum_node is None:
            raise RuntimeError(
                "EXPECTED_TEMPLATE_SHA256 "
                "was not found."
            )

        driver_lines = (
            driver_source.splitlines(
                keepends=True
            )
        )

        driver_lines[
            checksum_node.lineno - 1:
            checksum_node.end_lineno
        ] = [
            (
                "EXPECTED_TEMPLATE_SHA256 = "
                + repr(template_checksums)
                + chr(10)
            )
        ]

        driver_source = "".join(
            driver_lines
        )

        wrapper_marker = (
            "for product, template_path in "
            "[('population', POPULATION_TEMPLATE), "
            "('smod', SMOD_TEMPLATE), "
            "('built_surface', BUILT_TEMPLATE)]:"
        )

        if (
            "_original_create_enrichment_program"
            not in driver_source
        ):
            driver_source = insert_before_once(
                driver_source,
                wrapper_marker,
                driver_wrapper,
                "Enrichment wrapper insertion",
            )

        compile(
            driver_source,
            str(DRIVER),
            "exec",
        )

        DRIVER.write_text(
            driver_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(DRIVER),
            doraise=True,
        )


        # ====================================================
        # REFRESH WORKSPACE MANIFEST
        # ====================================================

        manifest = json.loads(
            WORKSPACE_MANIFEST.read_text(
                encoding="utf-8"
            )
        )

        manifest[
            "driver_sha256"
        ] = sha256_file(
            DRIVER
        )

        manifest[
            "template_sha256"
        ] = template_checksums

        manifest[
            "coastal_grid_patch"
        ] = {
            "version": PATCH_VERSION,
            "canonical_grid_cells": 4900,
            "population_no_coverage_value": 0.0,
            "smod_no_coverage_class": 0,
            "built_no_coverage_value": 0.0,
            "coverage_indicators": True,
            "training_filter_required": True,
            "applied_at_utc": time.strftime(
                "%Y-%m-%dT%H:%M:%SZ",
                time.gmtime(),
            ),
        }

        WORKSPACE_MANIFEST.write_text(
            json.dumps(
                manifest,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


        # ====================================================
        # RESET GENERATED PROGRAMS
        # ====================================================

        if PROGRAMS_ROOT.exists():
            shutil.rmtree(
                PROGRAMS_ROOT
            )

        PROGRAMS_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )


        # ====================================================
        # PRESERVE FAILED LOG AND REMOVE STALE PID
        # ====================================================

        log_preserved = False

        if LOG_FILE.is_file():
            failed_log = (
                ROOT
                / "batch"
                / (
                    "city.log.before_coastal_patch."
                    + str(int(time.time()))
                )
            )

            LOG_FILE.replace(
                failed_log
            )

            log_preserved = True

        if PID_FILE.is_file():
            PID_FILE.unlink()


        # ====================================================
        # REMOVE GENERATED BYTECODE
        # ====================================================

        for pycache in ROOT.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )


        # ====================================================
        # FINAL STATIC VERIFICATION
        # ====================================================

        for source_file in [
            POPULATION_TEMPLATE,
            SMOD_TEMPLATE,
            BUILT_TEMPLATE,
            DRIVER,
        ]:
            compile(
                source_file.read_text(
                    encoding="utf-8"
                ),
                str(source_file),
                "exec",
            )

        final_driver_tree = ast.parse(
            DRIVER.read_text(
                encoding="utf-8"
            )
        )

        final_checksums = None

        for node in final_driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target in node.targets
            ):
                final_checksums = (
                    ast.literal_eval(
                        node.value
                    )
                )

                break

        if final_checksums != template_checksums:
            raise RuntimeError(
                "Driver template checksums do not "
                "match the patched templates."
            )


    except Exception:
        restore_backup()

        if PROGRAMS_ROOT.exists():
            shutil.rmtree(
                PROGRAMS_ROOT
            )

        PROGRAMS_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )

        raise


    print("=" * 80)
    print("CHENNAI COASTAL CANONICAL-GRID CORRECTION")
    print("=" * 80)

    print(
        "Backup:",
        BACKUP_ROOT,
    )

    print()
    print("Canonical grid cells: 4900")
    print("Population no-coverage value: 0.0")
    print("SMOD no-coverage class: 0")
    print("Built no-coverage value: 0.0")
    print("Population coverage indicator: ENABLED")
    print("SMOD coverage indicator: ENABLED")
    print("Built coverage indicator: ENABLED")
    print("Enrichment completeness logic: PATCHED")
    print("Template checksums refreshed: PASS")
    print("Workspace manifest refreshed: PASS")
    print("Generated programs reset: PASS")
    print(
        "Failed log preserved:",
        log_preserved,
    )
    print("Stale PID removed: PASS")
    print("Python compilation: PASS")
    print("HDFS modified: NO")
    print("Spark launched: NO")
    print(
        "CHENNAI COASTAL GRID CORRECTION: SUCCESS"
    )
    '''
).strip() + "\n"


# ============================================================
# VALIDATE AND ENCODE REMOTE PATCHER
# ============================================================

compile(
    remote_patcher,
    "patch_chennai_coastal_grid.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print(
    "LOCAL CHENNAI COASTAL PATCHER COMPILE: PASS"
)

print(
    "Expected patcher SHA-256:",
    patcher_sha256,
)


patcher_b64 = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

remote_command = f"""
set -e

PATCHER="/tmp/patch_chennai_coastal_grid.py"
PAYLOAD="/tmp/chennai_coastal_patch_payload.json"
ROOT="{root}"

EXPECTED_PATCHER_SHA256="{patcher_sha256}"
EXPECTED_PAYLOAD_SHA256="{payload_sha256}"

echo "{patcher_b64}" \
| base64 -d \
> "${{PATCHER}}"

echo "{payload_b64}" \
| base64 -d \
> "${{PAYLOAD}}"

ACTUAL_PATCHER_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

ACTUAL_PAYLOAD_SHA256="$(
    sha256sum "${{PAYLOAD}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_PATCHER_SHA256}}"
echo "Remote payload SHA-256: ${{ACTUAL_PAYLOAD_SHA256}}"

if [ "${{ACTUAL_PATCHER_SHA256}}" != "${{EXPECTED_PATCHER_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

if [ "${{ACTUAL_PAYLOAD_SHA256}}" != "${{EXPECTED_PAYLOAD_SHA256}}" ]; then
    echo "ERROR: Payload checksum mismatch."
    exit 21
fi

echo "REMOTE PATCHER CHECKSUM: PASS"
echo "REMOTE PAYLOAD CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 \
    "${{PATCHER}}" \
    "${{ROOT}}" \
    "${{PAYLOAD}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 6G RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print("HDFS modified: NO")
print("Spark launched: NO")
print("CELL 6G: SUCCESS")

LOCAL CHENNAI COASTAL PATCHER COMPILE: PASS
Expected patcher SHA-256: 8fca9d3ed19377dece3e1c1157589a388f50240c68498e98a69b26eea6efc90e
Remote patcher SHA-256: 8fca9d3ed19377dece3e1c1157589a388f50240c68498e98a69b26eea6efc90e
Remote payload SHA-256: a73595b2a1c4366b7377fcd0230034f046584fe099b69992e0822d9fd13a33eb
REMOTE PATCHER CHECKSUM: PASS
REMOTE PAYLOAD CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
CHENNAI COASTAL CANONICAL-GRID CORRECTION
Backup: /home/ubuntu/ghsl_multicity/chennai_metro/coastal_patch_backup

Canonical grid cells: 4900
Population no-coverage value: 0.0
SMOD no-coverage class: 0
Built no-coverage value: 0.0
Population coverage indicator: ENABLED
SMOD coverage indicator: ENABLED
Built coverage indicator: ENABLED
Enrichment completeness logic: PATCHED
Template checksums refreshed: PASS
Workspace manifest refreshed: PASS
Generated programs reset: PASS
Failed log preserved: True
Stale PID removed: PASS
Python compilation: PASS
HDFS modified: NO
Spark launched: NO
CHENNAI C

In [22]:
# ============================================================
# CELL 6: ALL-EPOCH GENERATOR SMOKE TEST
#
# Generates and compiles:
#   10 population programs
#   10 SMOD programs
#   10 built-surface programs
#
# Total: 30 generated programs
#
# This cell does not:
#   - launch Spark
#   - execute generated programs
#   - read raster data
#   - write to HDFS
#   - modify the Chennai workspace
#
# Generated programs are written only to /tmp and deleted.
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


remote_smoke_test = r'''
set -e

ROOT="__CHENNAI_ROOT__"
DRIVER="$ROOT/batch/run_city_epochs.py"

echo "============================================================"
echo "CHENNAI ALL-EPOCH GENERATOR SMOKE TEST"
echo "============================================================"

if [ ! -f "$DRIVER" ]; then
    echo "ERROR: Chennai driver is missing:"
    echo "$DRIVER"
    exit 20
fi


python3 - "$ROOT" "$DRIVER" <<'PY'
from pathlib import Path
import ast
import py_compile
import shutil
import sys
import tempfile


root = Path(sys.argv[1])
driver_path = Path(sys.argv[2])

driver_source = driver_path.read_text(
    encoding="utf-8"
)

driver_tree = ast.parse(
    driver_source
)


# ============================================================
# BUILD A DEFINITIONS-ONLY DRIVER MODULE
#
# Keep imports, constants, assignments, and functions.
# Exclude the top-level production loop so no Spark job starts.
# ============================================================

allowed_node_types = (
    ast.Import,
    ast.ImportFrom,
    ast.Assign,
    ast.AnnAssign,
    ast.FunctionDef,
    ast.AsyncFunctionDef,
    ast.ClassDef,
)

definition_nodes = [
    node
    for node in driver_tree.body
    if isinstance(
        node,
        allowed_node_types,
    )
]

definition_module = ast.Module(
    body=definition_nodes,
    type_ignores=[],
)

ast.fix_missing_locations(
    definition_module
)


namespace = {
    "__file__": str(driver_path),
    "__name__": "chennai_generator_smoke_test",
}

exec(
    compile(
        definition_module,
        str(driver_path),
        "exec",
    ),
    namespace,
)


# ============================================================
# REQUIRED FACTORIES
# ============================================================

required_factories = {
    "population":
        "generate_population_program",
    "smod":
        "generate_smod_program",
    "built_surface":
        "generate_built_program",
}

factories = {}

for stage, function_name in (
    required_factories.items()
):
    factory = namespace.get(
        function_name
    )

    if not callable(factory):
        raise RuntimeError(
            "Required factory is unavailable: "
            + function_name
        )

    factories[stage] = factory


# ============================================================
# TEST CONFIGURATION
# ============================================================

epochs = [
    1975,
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
    2020,
]

expected_program_count = (
    len(epochs)
    * len(factories)
)

temporary_root = Path(
    tempfile.mkdtemp(
        prefix="chennai_generator_smoke_"
    )
)

generated_records = []


# ============================================================
# GENERATE AND COMPILE ALL 30 PROGRAMS
# ============================================================

try:
    for epoch in epochs:
        epoch_directory = (
            temporary_root
            / ("epoch_" + str(epoch))
        )

        epoch_directory.mkdir()

        for stage, factory in (
            factories.items()
        ):
            generated_object = factory(
                epoch,
                epoch_directory,
            )

            # The production factories normally return a Path.
            # Support source text as a defensive fallback.
            if isinstance(
                generated_object,
                Path,
            ):
                program_path = generated_object

            elif (
                isinstance(
                    generated_object,
                    str,
                )
                and "\n"
                not in generated_object
            ):
                program_path = Path(
                    generated_object
                )

            elif isinstance(
                generated_object,
                str,
            ):
                program_path = (
                    epoch_directory
                    / (
                        stage
                        + "_"
                        + str(epoch)
                        + ".py"
                    )
                )

                program_path.write_text(
                    generated_object,
                    encoding="utf-8",
                )

            else:
                raise RuntimeError(
                    "Unexpected factory return type "
                    "for "
                    + stage
                    + " "
                    + str(epoch)
                    + ": "
                    + type(
                        generated_object
                    ).__name__
                )


            if not program_path.is_file():
                raise RuntimeError(
                    "Factory did not create a program "
                    "for "
                    + stage
                    + " "
                    + str(epoch)
                    + ": "
                    + str(program_path)
                )


            # Compile every generated program.
            py_compile.compile(
                str(program_path),
                doraise=True,
            )


            program_source = (
                program_path.read_text(
                    encoding="utf-8"
                )
            )


            # --------------------------------------------
            # Chennai path validation
            # --------------------------------------------

            if (
                "city=chennai_metro"
                not in program_source
            ):
                raise RuntimeError(
                    "Chennai HDFS path is absent from "
                    + str(program_path)
                )

            if (
                "/home/ubuntu/ghsl_multicity/"
                "chennai_metro"
                not in program_source
                and stage
                in {
                    "population",
                    "smod",
                    "built_surface",
                }
            ):
                # This is informational in most generated
                # programs, but reject only if neither Chennai
                # workspace nor Chennai HDFS identity appears.
                if (
                    "chennai_metro"
                    not in program_source
                ):
                    raise RuntimeError(
                        "Chennai identity is absent from "
                        + str(program_path)
                    )


            # --------------------------------------------
            # Hyderabad leakage validation
            # --------------------------------------------

            forbidden_values = [
                "city=hyderabad",
                "/home/ubuntu/ghsl_multicity/"
                "hyderabad",
                "Hyderabad Metropolitan Region",
            ]

            remaining_forbidden = [
                value
                for value
                in forbidden_values
                if value in program_source
            ]

            if remaining_forbidden:
                raise RuntimeError(
                    "Hyderabad leakage in "
                    + str(program_path)
                    + ": "
                    + repr(
                        remaining_forbidden
                    )
                )


            # --------------------------------------------
            # Active epoch validation
            # --------------------------------------------

            expected_epoch_assignment = (
                "EPOCH = "
                + str(epoch)
            )

            compact_program_source = (
                program_source.replace(
                    " ",
                    ""
                )
            )

            compact_expected_assignment = (
                expected_epoch_assignment.replace(
                    " ",
                    ""
                )
            )

            if (
                compact_expected_assignment
                not in compact_program_source
            ):
                raise RuntimeError(
                    "Active epoch assignment is "
                    "incorrect in "
                    + str(program_path)
                    + ". Expected "
                    + expected_epoch_assignment
                )


            # --------------------------------------------
            # Expected HDFS epoch path validation
            # --------------------------------------------

            expected_epoch_path = (
                "epoch="
                + str(epoch)
                + "/"
            )

            if (
                expected_epoch_path
                not in program_source
            ):
                raise RuntimeError(
                    "Expected epoch path is absent "
                    "from "
                    + str(program_path)
                    + ": "
                    + expected_epoch_path
                )


            generated_records.append(
                {
                    "epoch": epoch,
                    "stage": stage,
                    "path": str(
                        program_path
                    ),
                }
            )

            print(
                "PASS:",
                epoch,
                stage,
                program_path.name,
            )


    # ========================================================
    # FINAL COUNTS
    # ========================================================

    if (
        len(generated_records)
        != expected_program_count
    ):
        raise RuntimeError(
            "Expected "
            + str(
                expected_program_count
            )
            + " generated programs, found "
            + str(
                len(generated_records)
            )
        )


    stage_counts = {}

    for stage in factories:
        stage_counts[stage] = sum(
            1
            for record
            in generated_records
            if record["stage"]
            == stage
        )


    expected_stage_count = len(
        epochs
    )

    for stage, count in (
        stage_counts.items()
    ):
        if count != expected_stage_count:
            raise RuntimeError(
                "Expected "
                + str(
                    expected_stage_count
                )
                + " "
                + stage
                + " programs, found "
                + str(count)
            )


    print()
    print(
        "Population programs:",
        stage_counts["population"],
    )

    print(
        "SMOD programs:",
        stage_counts["smod"],
    )

    print(
        "Built-surface programs:",
        stage_counts[
            "built_surface"
        ],
    )

    print(
        "Total generated programs:",
        len(generated_records),
    )

    print()
    print(
        "ALL_30_GENERATED_PROGRAMS_COMPILE_PASS"
    )


finally:
    # No generated smoke-test artifacts persist.
    if temporary_root.exists():
        shutil.rmtree(
            temporary_root
        )


print(
    "Temporary smoke-test directory removed: PASS"
)

print(
    "ALL_EPOCH_GENERATOR_SMOKE_TEST_SUCCESS"
)
PY


echo
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "Chennai workspace modified: NO"
echo "CELL_6_REMOTE_SUCCESS"
'''


# Replace a literal placeholder.
# Do not wrap the embedded Python in an outer f-string.

cmd = remote_smoke_test.replace(
    "__CHENNAI_ROOT__",
    root,
)


smoke_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 6 RESULT")
print("=" * 80)

print(
    "Return code:",
    smoke_result.returncode,
)

print(
    "Expected generated programs:",
    30,
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("Chennai workspace modified: NO")
print("CELL 6: SUCCESS")

CHENNAI ALL-EPOCH GENERATOR SMOKE TEST
PASS: 1975 population population_1975.py
PASS: 1975 smod smod_1975.py
PASS: 1975 built_surface built_surface_1975.py
PASS: 1980 population population_1980.py
PASS: 1980 smod smod_1980.py
PASS: 1980 built_surface built_surface_1980.py
PASS: 1985 population population_1985.py
PASS: 1985 smod smod_1985.py
PASS: 1985 built_surface built_surface_1985.py
PASS: 1990 population population_1990.py
PASS: 1990 smod smod_1990.py
PASS: 1990 built_surface built_surface_1990.py
PASS: 1995 population population_1995.py
PASS: 1995 smod smod_1995.py
PASS: 1995 built_surface built_surface_1995.py
PASS: 2000 population population_2000.py
PASS: 2000 smod smod_2000.py
PASS: 2000 built_surface built_surface_2000.py
PASS: 2005 population population_2005.py
PASS: 2005 smod smod_2005.py
PASS: 2005 built_surface built_surface_2005.py
PASS: 2010 population population_2010.py
PASS: 2010 smod smod_2010.py
PASS: 2010 built_surface built_surface_2010.py
PASS: 2015 population pop

In [23]:
# ============================================================
# CELL 7: DETACHED CHENNAI PRODUCTION LAUNCH
#
# Launches the full 1975-2020 Chennai production pipeline.
#
# Safety:
#   - Refuses a duplicate active launch
#   - Loads Hadoop/Sedona environment
#   - Verifies HDFS daemons and safe mode
#   - Verifies Chennai driver and template compilation
#   - Writes PID before returning
#   - Detects immediate startup failure
#   - Leaves the job detached on EC2
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


remote_launch = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

ROOT="__CHENNAI_ROOT__"
DRIVER="$ROOT/batch/run_city_epochs.py"
PID_FILE="$ROOT/batch/city.pid"
LOG_FILE="$ROOT/batch/city.log"


echo "============================================================"
echo "CHENNAI PRODUCTION LAUNCH"
echo "============================================================"


# ------------------------------------------------------------
# Verify HDFS services.
# ------------------------------------------------------------

for daemon in \
    NameNode \
    DataNode \
    SecondaryNameNode

do
    jps \
        | awk '{print $2}' \
        | grep -qx "$daemon"
done

hdfs dfsadmin -safemode get \
    | grep -qi "Safe mode is OFF"

echo "HDFS services: PASS"


# ------------------------------------------------------------
# Verify Chennai production files.
# ------------------------------------------------------------

if [ ! -f "$DRIVER" ]; then
    echo "ERROR: Chennai driver is missing."
    exit 20
fi

python3 -m py_compile "$DRIVER"

for template in "$ROOT"/templates/*.py; do
    python3 -m py_compile "$template"
done

echo "Chennai Python compilation: PASS"


# ------------------------------------------------------------
# Refuse duplicate launch.
# ------------------------------------------------------------

if [ -f "$PID_FILE" ]; then
    OLD_PID="$(cat "$PID_FILE")"

    if kill -0 "$OLD_PID" 2>/dev/null; then
        echo "ERROR: Chennai job is already running."
        echo "Active PID: $OLD_PID"
        exit 30
    fi

    echo "Removing stale PID file:"
    echo "$PID_FILE"

    rm -f "$PID_FILE"
fi


# ------------------------------------------------------------
# Preserve an old log if one unexpectedly exists.
# ------------------------------------------------------------

if [ -f "$LOG_FILE" ]; then
    LOG_BACKUP="$LOG_FILE.before_launch.$(
        date -u +%Y%m%dT%H%M%SZ
    )"

    mv "$LOG_FILE" "$LOG_BACKUP"

    echo "Previous log moved to:"
    echo "$LOG_BACKUP"
fi


# ------------------------------------------------------------
# Launch detached production driver.
# ------------------------------------------------------------

nohup python3 "$DRIVER" \
    >"$LOG_FILE" \
    2>&1 \
    </dev/null &

NEW_PID="$!"

echo "$NEW_PID" > "$PID_FILE"

echo "New PID: $NEW_PID"
echo "Log: $LOG_FILE"


# ------------------------------------------------------------
# Allow startup checks to run.
# ------------------------------------------------------------

sleep 5


# ------------------------------------------------------------
# Confirm process survived startup.
# ------------------------------------------------------------

if ! kill -0 "$NEW_PID" 2>/dev/null; then
    echo
    echo "ERROR: Chennai driver exited during startup."

    echo
    echo "Latest 100 log lines:"

    if [ -f "$LOG_FILE" ]; then
        tail -n 100 "$LOG_FILE"
    fi

    exit 31
fi


echo
ps -p "$NEW_PID" \
    -o pid,etime,%cpu,%mem,cmd

echo
echo "DETACHED_LAUNCH_SUCCESS"
'''


cmd = remote_launch.replace(
    "__CHENNAI_ROOT__",
    root,
)


launch_result = run_remote(
    cmd,
    timeout=90,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7 RESULT")
print("=" * 80)

print(
    "Return code:",
    launch_result.returncode,
)

print("Safe to close Colab.")
print("Monitor with Cell 8.")
print("CELL 7: SUCCESS")

CHENNAI PRODUCTION LAUNCH
HDFS services: PASS
Chennai Python compilation: PASS
New PID: 187627
Log: /home/ubuntu/ghsl_multicity/chennai_metro/batch/city.log

    PID     ELAPSED %CPU %MEM CMD
 187627       00:05  0.6  0.2 python3 /home/ubuntu/ghsl_multicity/chennai_metro/batch/run_city_epochs.py

DETACHED_LAUNCH_SUCCESS

REMOTE STDERR:


CELL 7 RESULT
Return code: 0
Safe to close Colab.
Monitor with Cell 8.
CELL 7: SUCCESS


In [42]:
# ============================================================
# CELL 8: STABLE CHENNAI MONITOR WITH CLEAR FINAL STATUS
#
# Read-only:
#   - Does not launch a job
#   - Does not stop a job
#   - Does not modify HDFS
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]
city_name = ACTIVE_CITY["city_name"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


remote_monitor = r'''
set +e

source /home/ubuntu/ghsl-sedona-env.sh 2>/dev/null

export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

ROOT="__CHENNAI_ROOT__"
CITY_NAME="__CITY_NAME__"

PID_FILE="$ROOT/batch/city.pid"
LOG_FILE="$ROOT/batch/city.log"
MANIFEST="$ROOT/batch/batch_manifest.tsv"


get_job_status() {
    RECORDED_PID="N/A"
    JOB_STATUS="UNKNOWN"

    if [ -f "$PID_FILE" ]; then
        RECORDED_PID="$(cat "$PID_FILE")"

        if kill -0 "$RECORDED_PID" 2>/dev/null; then
            JOB_STATUS="RUNNING"
        else
            JOB_STATUS="STOPPED"
        fi
    fi
}


# ============================================================
# INITIAL STATUS
# ============================================================

get_job_status

echo "============================================================"
echo "CHENNAI JOB STATUS"
echo "============================================================"

echo "City: $CITY_NAME"
echo "Recorded PID: $RECORDED_PID"
echo "STATUS=$JOB_STATUS"

if [ "$JOB_STATUS" = "RUNNING" ]; then
    ps -p "$RECORDED_PID" \
        -o pid,etime,%cpu,%mem,cmd
fi


# ============================================================
# LATEST LOG
# ============================================================

echo
echo "============================================================"
echo "LATEST 150 LOG LINES"
echo "============================================================"

if [ -f "$LOG_FILE" ]; then
    tail -n 150 "$LOG_FILE"
else
    echo "Log file is missing."
fi


# ============================================================
# CURRENT MANIFEST
# ============================================================

echo
echo "============================================================"
echo "CURRENT CHENNAI MANIFEST"
echo "============================================================"

if [ -f "$MANIFEST" ]; then
    cat "$MANIFEST"
else
    echo "Manifest has not been created."
fi


# ============================================================
# SIMPLE PROGRESS COUNTS
# ============================================================

echo
echo "============================================================"
echo "CURRENT PROGRESS SUMMARY"
echo "============================================================"

COMPLETED_STAGES=0
COMPLETED_EPOCHS=0

if [ -f "$MANIFEST" ]; then
    COMPLETED_STAGES="$(
        grep -E \
            $'\t(built_and_validated|existing_valid_skipped)\t' \
            "$MANIFEST" \
            | wc -l \
            | tr -d ' '
    )"

    for epoch in \
        1975 1980 1985 1990 1995 \
        2000 2005 2010 2015 2020
    do
        EPOCH_RECORDS="$(
            awk -F '\t' \
                -v target_epoch="$epoch" \
                '$1 == target_epoch {print}' \
                "$MANIFEST" \
                | grep -E \
                    $'\t(built_and_validated|existing_valid_skipped)\t' \
                | wc -l \
                | tr -d ' '
        )"

        if [ "$EPOCH_RECORDS" = "4" ]; then
            COMPLETED_EPOCHS="$(
                expr "$COMPLETED_EPOCHS" + 1
            )"
        fi
    done
fi

echo "Completed stage records: $COMPLETED_STAGES / 40"
echo "Fully completed epochs: $COMPLETED_EPOCHS / 10"


# ============================================================
# ACTIVE JAVA PROCESSES
# ============================================================

echo
echo "============================================================"
echo "ACTIVE SPARK / JAVA PROCESSES"
echo "============================================================"

jps


# ============================================================
# FINAL STATUS
# ============================================================

get_job_status

COMPLETION_RESULT="UNKNOWN"
FINAL_MESSAGE="CHENNAI JOB STATUS UNKNOWN"

if [ "$JOB_STATUS" = "RUNNING" ]; then
    COMPLETION_RESULT="IN_PROGRESS"
    FINAL_MESSAGE="CHENNAI JOB IS STILL RUNNING"
fi

if [ "$JOB_STATUS" = "STOPPED" ]; then
    if [ -f "$LOG_FILE" ]; then
        if grep -q "BATCH PIPELINE: SUCCESS" "$LOG_FILE"; then
            COMPLETION_RESULT="SUCCESS"
            FINAL_MESSAGE="CHENNAI JOB COMPLETED SUCCESSFULLY"
        elif grep -q "Traceback (most recent call last)" "$LOG_FILE"; then
            COMPLETION_RESULT="FAILED"
            FINAL_MESSAGE="CHENNAI JOB STOPPED WITH FAILURE"
        else
            COMPLETION_RESULT="UNKNOWN"
            FINAL_MESSAGE="CHENNAI JOB STOPPED, RESULT UNKNOWN"
        fi
    fi
fi


# ============================================================
# FINAL BLOCK
# ============================================================

echo
echo
echo "############################################################"
echo "############################################################"
echo
echo "              FINAL CHENNAI PIPELINE STATUS"
echo
echo "City: $CITY_NAME"
echo "PID: $RECORDED_PID"
echo "FINAL_STATUS=$JOB_STATUS"
echo "COMPLETION_RESULT=$COMPLETION_RESULT"
echo "COMPLETED_STAGES=$COMPLETED_STAGES/40"
echo "COMPLETED_EPOCHS=$COMPLETED_EPOCHS/10"
echo
echo ">>> $FINAL_MESSAGE <<<"
echo

if [ "$JOB_STATUS" = "RUNNING" ]; then
    ps -p "$RECORDED_PID" \
        -o pid,etime,%cpu,%mem,cmd

    echo
    echo "Action: Let the EC2 job continue."
    echo "Do not rerun Cell 7."
fi

if [ "$COMPLETION_RESULT" = "SUCCESS" ]; then
    echo "Action: Run Cell 9 final audit."
fi

if [ "$COMPLETION_RESULT" = "FAILED" ]; then
    echo "Latest failure markers:"

    grep -n \
        -E \
        "Traceback|RuntimeError|AnalysisException|Return code: 1" \
        "$LOG_FILE" \
        | tail -n 20

    echo
    echo "Action: Do not rerun Cell 7."
fi

echo
echo "############################################################"
echo "############################################################"

exit 0
'''


cmd = (
    remote_monitor
    .replace(
        "__CHENNAI_ROOT__",
        root,
    )
    .replace(
        "__CITY_NAME__",
        city_name,
    )
)


monitor_result = run_remote(
    cmd,
    timeout=180,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 8 RESULT")
print("=" * 80)

print(
    "Monitor return code:",
    monitor_result.returncode,
)

print("No new job was launched.")
print("No running process was stopped.")
print("No HDFS data was modified.")

CHENNAI JOB STATUS
City: Chennai Metropolitan Region
Recorded PID: 187627
STATUS=STOPPED

LATEST 150 LOG LINES
VALIDATE NEW ENRICHED 2015
Command: spark-submit --master local[1] --driver-memory 1g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=2 --conf spark.default.parallelism=2 --conf spark.sql.adaptive.enabled=false --conf spark.sql.codegen.wholeStage=false --conf spark.driver.maxResultSize=256m /home/ubuntu/ghsl_multicity/chennai_metro/programs/epoch_2015/validate_enriched_2015.py


Return code: 0
Elapsed seconds: 10.63
enriched 2015: BUILT AND VALIDATED

EPOCH 2015: COMPLETE
Epoch elapsed minutes: 8.14

################################################################################
EPOCH 2020 (10/10)
################################################################################

RUN POPULATION 2020
Command: spark-submit --master local[1] --driver-memory 5g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=4 --conf spark.default.parallelism=2 -

In [43]:
# CELL 9: FINAL 40-PATH AUDIT
root=ACTIVE_CITY['remote_work_dir']; slug=ACTIVE_CITY_SLUG
cmd=f"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH=${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}
grep -q 'BATCH PIPELINE: SUCCESS' {root}/batch/city.log
n=0
for e in 1975 1980 1985 1990 1995 2000 2005 2010 2015 2020; do
 for s in population smod built_surface; do hdfs dfs -test -e /data/ghsl/curated/cities/city={slug}/product=$s/epoch=$e/grid_1km; n=$((n+1)); done
 hdfs dfs -test -e /data/ghsl/curated/cities/city={slug}/enriched_grid_year/epoch=$e; n=$((n+1)); echo Epoch $e: 4/4 paths PASS
done
test $n -eq 40; echo ALL_40_PATHS_PASS; echo CITY_COMPLETE
"""
run_remote(cmd,timeout=300); print('CELL 9: SUCCESS')


Epoch 1975: 4/4 paths PASS
Epoch 1980: 4/4 paths PASS
Epoch 1985: 4/4 paths PASS
Epoch 1990: 4/4 paths PASS
Epoch 1995: 4/4 paths PASS
Epoch 2000: 4/4 paths PASS
Epoch 2005: 4/4 paths PASS
Epoch 2010: 4/4 paths PASS
Epoch 2015: 4/4 paths PASS
Epoch 2020: 4/4 paths PASS
ALL_40_PATHS_PASS
CITY_COMPLETE

REMOTE STDERR:

CELL 9: SUCCESS


The validated data is stored permanently under:

Plain Text
hdfs://localhost:9000/data/ghsl/curated/cities/city=chennai_metro/


The principal modelling datasets are:

Plain Text
hdfs://localhost:9000/data/ghsl/curated/cities/
city=chennai_metro/enriched_grid_year/epoch=<YEAR>
Important Chennai-specific note

Chennai retains all 4,900 canonical cells per epoch. Cells without valid raster coverage were assigned zero values and can be distinguished using the coverage indicators:

Plain Text
has_valid_population_coverage
has_valid_smod_coverage
has_valid_built_coverage

When training the model, filter or explicitly encode these flags so no-coverage coastal cells are not interpreted as observed zero-valued land.